<a href="https://colab.research.google.com/github/worasak2530seed/fast-hybrid-pro/blob/main/Fast_Hybrid_Pro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [42]:

# ============================================================
# FAST HYBRID PRO v2
# CLEAN RESEARCH / BACKTEST NOTEBOOK
# ============================================================

!pip -q install yfinance pandas numpy matplotlib

import warnings
warnings.filterwarnings("ignore")

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from dataclasses import dataclass, asdict

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 100)

print("✅ Setup complete")

✅ Setup complete


In [43]:

# ============================================================
# 2) CONFIGURATION
# ============================================================

CAPITAL = 100_000

# Maximum risk per trade
MAX_RISK_PCT = 0.02

# Thai board lot
BOARD_LOT = 100

# Minimum score for signal
MIN_SCORE = 3

# Historical data requirement
MIN_BARS = 200

# Backtest warm-up
WARMUP_BARS = 200

# Maximum holding period
MAX_HOLDING_DAYS = 20


WATCHLIST = sorted(set([
    'DELTA.BK',
    'SNNP.BK',
    'JMART.BK',
    'JMT.BK',
    'FORTH.BK',
    'SPA.BK',
    'AU.BK',
    'SAPPE.BK',
    'ICHI.BK',
    'CBG.BK',
    'KAMART.BK',
    'MOSHI.BK',
    'MEB.BK',
    'MASTER.BK',
    'KLINIQ.BK',
    'SNNP.BK',
    'TKN.BK',
    'XO.BK',
    'DITTO.BK',
    'PLANB.BK'
]))


@dataclass
class StrategyConfig:

    # --------------------------------------------------------
    # MACD
    # --------------------------------------------------------
    macd_fast: int = 4
    macd_slow: int = 12
    macd_signal: int = 9

    # --------------------------------------------------------
    # STOCHASTIC
    # --------------------------------------------------------
    stoch_k: int = 8
    stoch_smooth: int = 3
    stoch_d: int = 3

    # --------------------------------------------------------
    # DAILY TREND
    # --------------------------------------------------------
    ema_fast: int = 5
    ema_mid: int = 10
    ema_long: int = 75

    # --------------------------------------------------------
    # WEEKLY TREND
    # --------------------------------------------------------
    weekly_ema_fast: int = 5
    weekly_ema_slow: int = 10

    # --------------------------------------------------------
    # ATR
    # --------------------------------------------------------
    atr_length: int = 14
    atr_stop_mult: float = 1.5

    # --------------------------------------------------------
    # BOLLINGER
    # --------------------------------------------------------
    bb_length: int = 20
    bb_mult: float = 2.0

    # --------------------------------------------------------
    # KELTNER
    # --------------------------------------------------------
    kc_mult: float = 1.5

    # --------------------------------------------------------
    # SQUEEZE
    # --------------------------------------------------------
    squeeze_lookback: int = 10

    # --------------------------------------------------------
    # VOLUME
    # --------------------------------------------------------
    volume_ma: int = 5
    volume_ratio_min: float = 1.2

    # --------------------------------------------------------
    # TARGET
    # --------------------------------------------------------
    target1_pct: float = 0.07
    target2_pct: float = 0.15

    # --------------------------------------------------------
    # SCORE
    # --------------------------------------------------------
    min_score: int = 3


# Current canonical research setup
CANONICAL = StrategyConfig(
    macd_fast=4,
    macd_slow=12,
    macd_signal=9
)


# Alternative setup for comparison
FAST_MACD = StrategyConfig(
    macd_fast=6,
    macd_slow=13,
    macd_signal=5
)


print("WATCHLIST:", len(WATCHLIST))

print("\nCANONICAL")
print(asdict(CANONICAL))

print("\nFAST MACD")
print(asdict(FAST_MACD))

WATCHLIST: 19

CANONICAL
{'macd_fast': 4, 'macd_slow': 12, 'macd_signal': 9, 'stoch_k': 8, 'stoch_smooth': 3, 'stoch_d': 3, 'ema_fast': 5, 'ema_mid': 10, 'ema_long': 75, 'weekly_ema_fast': 5, 'weekly_ema_slow': 10, 'atr_length': 14, 'atr_stop_mult': 1.5, 'bb_length': 20, 'bb_mult': 2.0, 'kc_mult': 1.5, 'squeeze_lookback': 10, 'volume_ma': 5, 'volume_ratio_min': 1.2, 'target1_pct': 0.07, 'target2_pct': 0.15, 'min_score': 3}

FAST MACD
{'macd_fast': 6, 'macd_slow': 13, 'macd_signal': 5, 'stoch_k': 8, 'stoch_smooth': 3, 'stoch_d': 3, 'ema_fast': 5, 'ema_mid': 10, 'ema_long': 75, 'weekly_ema_fast': 5, 'weekly_ema_slow': 10, 'atr_length': 14, 'atr_stop_mult': 1.5, 'bb_length': 20, 'bb_mult': 2.0, 'kc_mult': 1.5, 'squeeze_lookback': 10, 'volume_ma': 5, 'volume_ratio_min': 1.2, 'target1_pct': 0.07, 'target2_pct': 0.15, 'min_score': 3}


In [44]:

# ============================================================
# 3) DATA ENGINE
# ============================================================

REQUIRED_COLS = [
    "Open",
    "High",
    "Low",
    "Close",
    "Volume"
]


def normalize_ohlcv(df):

    if df is None or df.empty:
        return pd.DataFrame()

    out = df.copy()

    # yfinance can return MultiIndex columns
    if isinstance(out.columns, pd.MultiIndex):
        out.columns = out.columns.get_level_values(0)

    # Remove duplicate columns
    out = out.loc[:, ~out.columns.duplicated()]

    missing = [
        c for c in REQUIRED_COLS
        if c not in out.columns
    ]

    if missing:
        raise ValueError(
            f"Missing OHLCV columns: {missing}"
        )

    out = out[REQUIRED_COLS].copy()

    # Datetime
    out.index = pd.to_datetime(out.index)

    # Remove timezone if present
    if getattr(out.index, "tz", None) is not None:
        out.index = out.index.tz_localize(None)

    # Numeric conversion
    for col in REQUIRED_COLS:
        out[col] = pd.to_numeric(
            out[col],
            errors="coerce"
        )

    # Remove invalid rows
    out = out.dropna(
        subset=[
            "Open",
            "High",
            "Low",
            "Close"
        ]
    )

    # Remove duplicate dates
    out = out[
        ~out.index.duplicated(
            keep="last"
        )
    ]

    # Sort chronologically
    out = out.sort_index()

    return out


def download_stock(
    symbol,
    period="5y"
):

    raw = yf.download(
        symbol,
        period=period,
        interval="1d",
        auto_adjust=False,
        progress=False,
        threads=False
    )

    return normalize_ohlcv(raw)


data = {}
errors = {}


print("🚀 Downloading market data...\n")


for symbol in WATCHLIST:

    try:

        df = download_stock(symbol)

        if len(df) < MIN_BARS:

            errors[symbol] = (
                f"Only {len(df)} bars "
                f"(minimum {MIN_BARS})"
            )

        else:

            data[symbol] = df

            print(
                f"✅ {symbol:<12} "
                f"{len(df):>5} bars"
            )

    except Exception as e:

        errors[symbol] = str(e)

        print(
            f"❌ {symbol}: {e}"
        )


print("\n====================================")
print(
    f"Loaded: {len(data)} / "
    f"{len(WATCHLIST)}"
)
print("====================================")


if errors:

    print("\n⚠️ Errors / insufficient data:")

    for symbol, error in errors.items():

        print(
            f"{symbol}: {error}"
        )

🚀 Downloading market data...

✅ AU.BK         1219 bars
✅ CBG.BK        1219 bars
✅ DELTA.BK      1219 bars
✅ DITTO.BK      1219 bars
✅ FORTH.BK      1219 bars
✅ ICHI.BK       1219 bars
✅ JMART.BK      1219 bars
✅ JMT.BK        1219 bars
✅ KAMART.BK     1219 bars
✅ KLINIQ.BK      958 bars
✅ MASTER.BK      904 bars
✅ MEB.BK         890 bars
✅ MOSHI.BK       927 bars
✅ PLANB.BK      1219 bars
✅ SAPPE.BK      1219 bars
✅ SNNP.BK       1219 bars
✅ SPA.BK        1219 bars
✅ TKN.BK        1219 bars
✅ XO.BK         1219 bars

Loaded: 19 / 19


In [45]:

# ============================================================
# 4) DAILY INDICATORS
# ============================================================

def wilder_rma(series, length):
    """
    Wilder RMA
    ใช้แนวคิดเดียวกับ TradingView ta.rma()
    """

    return series.ewm(
        alpha=1 / length,
        adjust=False,
        min_periods=length
    ).mean()


def calculate_atr(df, length=14):

    prev_close = df["Close"].shift(1)

    tr1 = df["High"] - df["Low"]
    tr2 = (df["High"] - prev_close).abs()
    tr3 = (df["Low"] - prev_close).abs()

    true_range = pd.concat(
        [tr1, tr2, tr3],
        axis=1
    ).max(axis=1)

    atr = wilder_rma(
        true_range,
        length
    )

    return true_range, atr


def calculate_daily_indicators(df):

    out = df.copy()

    close = out["Close"]
    high = out["High"]
    low = out["Low"]
    volume = out["Volume"]

    # --------------------------------------------------------
    # EMA 5 / 10 / 75
    # --------------------------------------------------------

    out["EMA5"] = close.ewm(
        span=5,
        adjust=False
    ).mean()

    out["EMA10"] = close.ewm(
        span=10,
        adjust=False
    ).mean()

    out["EMA75"] = close.ewm(
        span=75,
        adjust=False
    ).mean()

    # --------------------------------------------------------
    # MACD 4 / 12 / 9
    # --------------------------------------------------------

    ema_fast = close.ewm(
        span=4,
        adjust=False
    ).mean()

    ema_slow = close.ewm(
        span=12,
        adjust=False
    ).mean()

    out["MACD"] = ema_fast - ema_slow

    out["MACD_SIGNAL"] = out["MACD"].ewm(
        span=9,
        adjust=False
    ).mean()

    out["MACD_HIST"] = (
        out["MACD"] - out["MACD_SIGNAL"]
    )

    # --------------------------------------------------------
    # STOCHASTIC 8 / 3 / 3
    # --------------------------------------------------------

    lowest_8 = low.rolling(
        window=8,
        min_periods=8
    ).min()

    highest_8 = high.rolling(
        window=8,
        min_periods=8
    ).max()

    price_range = highest_8 - lowest_8

    stoch_raw = (
        100
        * (close - lowest_8)
        / price_range.replace(0, np.nan)
    )

    out["STOCH_K"] = stoch_raw.rolling(
        window=3,
        min_periods=3
    ).mean()

    out["STOCH_D"] = out["STOCH_K"].rolling(
        window=3,
        min_periods=3
    ).mean()

    # --------------------------------------------------------
    # ATR 14 — WILDER
    # --------------------------------------------------------

    out["TR"], out["ATR14"] = calculate_atr(
        out,
        length=14
    )

    # --------------------------------------------------------
    # BOLLINGER BAND 20 / 2
    # --------------------------------------------------------

    bb_mid = close.rolling(
        window=20,
        min_periods=20
    ).mean()

    bb_std = close.rolling(
        window=20,
        min_periods=20
    ).std()

    out["BB_MID"] = bb_mid

    out["BB_UPPER"] = (
        bb_mid + (2.0 * bb_std)
    )

    out["BB_LOWER"] = (
        bb_mid - (2.0 * bb_std)
    )

    # --------------------------------------------------------
    # KELTNER CHANNEL
    # --------------------------------------------------------

    out["KC_MID"] = bb_mid

    out["KC_UPPER"] = (
        bb_mid + (1.5 * out["ATR14"])
    )

    out["KC_LOWER"] = (
        bb_mid - (1.5 * out["ATR14"])
    )

    # --------------------------------------------------------
    # SQUEEZE
    # --------------------------------------------------------

    out["SQUEEZE"] = (
        (out["BB_UPPER"] <= out["KC_UPPER"])
        &
        (out["BB_LOWER"] >= out["KC_LOWER"])
    )

    # --------------------------------------------------------
    # SQUEEZE RELEASE
    # --------------------------------------------------------

    out["SQUEEZE_RELEASE"] = (
        (~out["SQUEEZE"])
        &
        (out["SQUEEZE"].shift(1).fillna(False))
    )

    # --------------------------------------------------------
    # BARS SINCE LAST SQUEEZE
    # --------------------------------------------------------

    bars_since_squeeze = []

    last_squeeze_index = None

    for i, is_squeeze in enumerate(
        out["SQUEEZE"].fillna(False)
    ):

        if is_squeeze:

            last_squeeze_index = i
            bars_since_squeeze.append(0)

        elif last_squeeze_index is None:

            bars_since_squeeze.append(np.nan)

        else:

            bars_since_squeeze.append(
                i - last_squeeze_index
            )

    out["BARS_SINCE_SQUEEZE"] = (
        bars_since_squeeze
    )

    out["SQUEEZE_RECENT"] = (
        out["BARS_SINCE_SQUEEZE"] <= 10
    )

    # --------------------------------------------------------
    # VOLUME
    # --------------------------------------------------------

    out["VOLUME_MA5"] = volume.rolling(
        window=5,
        min_periods=5
    ).mean()

    out["VOLUME_RATIO"] = (
        volume
        / out["VOLUME_MA5"].replace(0, np.nan)
    )

    # --------------------------------------------------------
    # CANDLE BODY
    # --------------------------------------------------------

    out["BODY"] = (
        close - out["Open"]
    ).abs()

    out["BODY_ATR_RATIO"] = (
        out["BODY"]
        / out["ATR14"].replace(0, np.nan)
    )

    return out


# ============================================================
# APPLY TO ALL SYMBOLS
# ============================================================

daily_data = {}
indicator_errors = {}

print("📊 Calculating daily indicators...\n")

for symbol, df in data.items():

    try:

        result = calculate_daily_indicators(df)

        daily_data[symbol] = result

        print(
            f"✅ {symbol:<12} "
            f"{len(result):>5} bars"
        )

    except Exception as e:

        indicator_errors[symbol] = str(e)

        print(
            f"❌ {symbol}: {e}"
        )


print("\n====================================")
print(
    f"Daily indicator data: "
    f"{len(daily_data)} / {len(data)}"
)
print("====================================")


if indicator_errors:

    print("\n⚠️ Indicator errors:")

    for symbol, error in indicator_errors.items():

        print(
            f"{symbol}: {error}"
        )


# ============================================================
# QUICK VALIDATION
# ============================================================

if daily_data:

    sample_symbol = next(iter(daily_data))

    sample = daily_data[sample_symbol]

    print(
        f"\n🔎 Quick validation: {sample_symbol}"
    )

    display(
        sample[
            [
                "Close",
                "EMA5",
                "EMA10",
                "EMA75",
                "MACD",
                "MACD_SIGNAL",
                "MACD_HIST",
                "STOCH_K",
                "STOCH_D",
                "ATR14",
                "SQUEEZE",
                "SQUEEZE_RELEASE",
                "VOLUME_RATIO"
            ]
        ].tail(5)
    )

📊 Calculating daily indicators...

✅ AU.BK         1219 bars
✅ CBG.BK        1219 bars
✅ DELTA.BK      1219 bars
✅ DITTO.BK      1219 bars
✅ FORTH.BK      1219 bars
✅ ICHI.BK       1219 bars
✅ JMART.BK      1219 bars
✅ JMT.BK        1219 bars
✅ KAMART.BK     1219 bars
✅ KLINIQ.BK      958 bars
✅ MASTER.BK      904 bars
✅ MEB.BK         890 bars
✅ MOSHI.BK       927 bars
✅ PLANB.BK      1219 bars
✅ SAPPE.BK      1219 bars
✅ SNNP.BK       1219 bars
✅ SPA.BK        1219 bars
✅ TKN.BK        1219 bars
✅ XO.BK         1219 bars

Daily indicator data: 19 / 19

🔎 Quick validation: AU.BK


Price,Close,EMA5,EMA10,EMA75,MACD,MACD_SIGNAL,MACD_HIST,STOCH_K,STOCH_D,ATR14,SQUEEZE,SQUEEZE_RELEASE,VOLUME_RATIO
Date,,,,,,,,,,,,,
2026-09-30,4.16,4.192573,4.211469,4.446661,-0.030768,-0.025483,-0.005285,26.666540,24.999877,0.042451,True,False,1.524227
2026-10-01,4.14,4.175048,4.198474,4.438591,-0.037745,-0.027935,-0.009810,21.666578,24.999886,0.047991,False,True,1.063792
2026-10-02,4.14,4.163366,4.187843,4.430733,-0.038964,-0.030141,-0.008823,16.666617,21.666578,0.051706,False,False,1.187989
2026-10-05,4.20,4.175577,4.190053,4.424662,-0.022416,-0.028596,0.006180,40.476141,26.269779,0.055155,True,False,1.358750
2026-10-06,4.18,4.177051,4.188225,4.418223,-0.017559,-0.026389,0.008830,51.190451,36.111070,0.054073,False,True,1.037242


In [46]:

# ============================================================
# 5) WEEKLY INDICATORS
# ============================================================

def calculate_weekly_indicators(df):

    daily = df.copy()

    # --------------------------------------------------------
    # Daily -> Weekly
    #
    # Week ends on Friday.
    # --------------------------------------------------------

    weekly = daily.resample("W-FRI").agg({
        "Open": "first",
        "High": "max",
        "Low": "min",
        "Close": "last",
        "Volume": "sum"
    })

    weekly = weekly.dropna(
        subset=[
            "Open",
            "High",
            "Low",
            "Close"
        ]
    )

    close = weekly["Close"]

    # --------------------------------------------------------
    # WEEKLY EMA 5 / 10
    # --------------------------------------------------------

    weekly["W_EMA5"] = close.ewm(
        span=5,
        adjust=False
    ).mean()

    weekly["W_EMA10"] = close.ewm(
        span=10,
        adjust=False
    ).mean()

    # --------------------------------------------------------
    # WEEKLY MACD 4 / 12 / 9
    # --------------------------------------------------------

    weekly_ema_fast = close.ewm(
        span=4,
        adjust=False
    ).mean()

    weekly_ema_slow = close.ewm(
        span=12,
        adjust=False
    ).mean()

    weekly["W_MACD"] = (
        weekly_ema_fast
        - weekly_ema_slow
    )

    weekly["W_MACD_SIGNAL"] = (
        weekly["W_MACD"].ewm(
            span=9,
            adjust=False
        ).mean()
    )

    weekly["W_MACD_HIST"] = (
        weekly["W_MACD"]
        - weekly["W_MACD_SIGNAL"]
    )

    # --------------------------------------------------------
    # IMPORTANT:
    #
    # A weekly candle ending Friday becomes available
    # only from the next trading day.
    #
    # Example:
    #
    # Week ending 2026-09-25
    # -> usable from 2026-09-28
    #
    # This prevents Friday from seeing its own completed
    # weekly candle during the Friday signal.
    # --------------------------------------------------------

    weekly_features = weekly[
        [
            "W_EMA5",
            "W_EMA10",
            "W_MACD",
            "W_MACD_SIGNAL",
            "W_MACD_HIST"
        ]
    ].copy()

    # Move availability date to next Monday.
    #
    # W-FRI + 3 days = Monday
    #
    # If Monday is a market holiday, forward-fill will keep
    # the last completed weekly value until the next trading day.
    weekly_features.index = (
        weekly_features.index
        + pd.Timedelta(days=3)
    )

    # --------------------------------------------------------
    # Map completed weekly values to daily data
    # --------------------------------------------------------

    daily_index = daily.index

    mapped_weekly = weekly_features.reindex(
        daily_index,
        method="ffill"
    )

    # --------------------------------------------------------
    # Combine
    # --------------------------------------------------------

    out = daily.copy()

    for col in mapped_weekly.columns:
        out[col] = mapped_weekly[col]

    return out, weekly


# ============================================================
# APPLY TO ALL SYMBOLS
# ============================================================

weekly_data = {}
combined_data = {}
weekly_errors = {}

print("📅 Calculating weekly indicators...\n")

for symbol, df in daily_data.items():

    try:

        combined, weekly = calculate_weekly_indicators(df)

        combined_data[symbol] = combined
        weekly_data[symbol] = weekly

        print(
            f"✅ {symbol:<12} "
            f"Daily: {len(combined):>5} | "
            f"Weekly: {len(weekly):>4}"
        )

    except Exception as e:

        weekly_errors[symbol] = str(e)

        print(
            f"❌ {symbol}: {e}"
        )


print("\n====================================")
print(
    f"Weekly indicator data: "
    f"{len(combined_data)} / {len(daily_data)}"
)
print("====================================")


if weekly_errors:

    print("\n⚠️ Weekly indicator errors:")

    for symbol, error in weekly_errors.items():

        print(
            f"{symbol}: {error}"
        )


# ============================================================
# VALIDATION
# ============================================================

if combined_data:

    sample_symbol = next(iter(combined_data))

    sample = combined_data[sample_symbol]

    print(
        f"\n🔎 Weekly validation: {sample_symbol}"
    )

    display(
        sample[
            [
                "Close",
                "W_EMA5",
                "W_EMA10",
                "W_MACD",
                "W_MACD_SIGNAL",
                "W_MACD_HIST"
            ]
        ].tail(15)
    )

📅 Calculating weekly indicators...

✅ AU.BK        Daily:  1219 | Weekly:  262
✅ CBG.BK       Daily:  1219 | Weekly:  262
✅ DELTA.BK     Daily:  1219 | Weekly:  262
✅ DITTO.BK     Daily:  1219 | Weekly:  262
✅ FORTH.BK     Daily:  1219 | Weekly:  262
✅ ICHI.BK      Daily:  1219 | Weekly:  262
✅ JMART.BK     Daily:  1219 | Weekly:  262
✅ JMT.BK       Daily:  1219 | Weekly:  262
✅ KAMART.BK    Daily:  1219 | Weekly:  262
✅ KLINIQ.BK    Daily:   958 | Weekly:  205
✅ MASTER.BK    Daily:   904 | Weekly:  194
✅ MEB.BK       Daily:   890 | Weekly:  191
✅ MOSHI.BK     Daily:   927 | Weekly:  199
✅ PLANB.BK     Daily:  1219 | Weekly:  262
✅ SAPPE.BK     Daily:  1219 | Weekly:  262
✅ SNNP.BK      Daily:  1219 | Weekly:  262
✅ SPA.BK       Daily:  1219 | Weekly:  262
✅ TKN.BK       Daily:  1219 | Weekly:  262
✅ XO.BK        Daily:  1219 | Weekly:  262

Weekly indicator data: 19 / 19

🔎 Weekly validation: AU.BK


Price,Close,W_EMA5,W_EMA10,W_MACD,W_MACD_SIGNAL,W_MACD_HIST
Date,,,,,,
2026-09-16,4.22,4.368172,4.466597,-0.152274,-0.090567,-0.061708
2026-09-17,4.24,4.368172,4.466597,-0.152274,-0.090567,-0.061708
2026-09-18,4.24,4.368172,4.466597,-0.152274,-0.090567,-0.061708
2026-09-21,4.24,4.325448,4.425397,-0.153826,-0.103219,-0.050607
2026-09-22,4.24,4.325448,4.425397,-0.153826,-0.103219,-0.050607
2026-09-23,4.22,4.325448,4.425397,-0.153826,-0.103219,-0.050607
2026-09-24,4.22,4.325448,4.425397,-0.153826,-0.103219,-0.050607
2026-09-25,4.20,4.325448,4.425397,-0.153826,-0.103219,-0.050607
2026-09-28,4.20,4.283632,4.384416,-0.154994,-0.113574,-0.041420


In [47]:

# ============================================================
# 6) BUILD FEATURE DATASETS
# ============================================================

feature_data = {}

feature_errors = {}


for symbol, df in combined_data.items():

    try:

        out = df.copy()

        # ----------------------------------------------------
        # Basic validity
        # ----------------------------------------------------

        required = [
            "Open",
            "High",
            "Low",
            "Close",
            "Volume",

            "EMA5",
            "EMA10",
            "EMA75",

            "MACD",
            "MACD_SIGNAL",
            "MACD_HIST",

            "STOCH_K",
            "STOCH_D",

            "ATR14",

            "BB_UPPER",
            "BB_LOWER",

            "KC_UPPER",
            "KC_LOWER",

            "SQUEEZE",
            "SQUEEZE_RELEASE",
            "SQUEEZE_RECENT",

            "VOLUME_RATIO",

            "W_EMA5",
            "W_EMA10",

            "W_MACD",
            "W_MACD_SIGNAL",
            "W_MACD_HIST"
        ]

        missing = [
            col
            for col in required
            if col not in out.columns
        ]

        if missing:

            raise ValueError(
                f"Missing columns: {missing}"
            )

        # ----------------------------------------------------
        # Remove rows that cannot produce indicators
        #
        # IMPORTANT:
        # We do NOT drop rows globally before calculating
        # indicators. All indicators have already been
        # calculated using the full historical dataset.
        # ----------------------------------------------------

        out = out.dropna(
            subset=[
                "EMA75",
                "ATR14",
                "MACD",
                "MACD_SIGNAL",
                "MACD_HIST",
                "STOCH_K",
                "STOCH_D",
                "W_EMA5",
                "W_EMA10",
                "W_MACD",
                "W_MACD_SIGNAL",
                "W_MACD_HIST",
                "VOLUME_RATIO"
            ]
        )

        # ----------------------------------------------------
        # Ensure chronological order
        # ----------------------------------------------------

        out = out.sort_index()

        # ----------------------------------------------------
        # Ensure unique dates
        # ----------------------------------------------------

        out = out[
            ~out.index.duplicated(
                keep="last"
            )
        ]

        # ----------------------------------------------------
        # Create previous-bar values
        #
        # Used later for signal confirmation.
        # ----------------------------------------------------

        out["PREV_CLOSE"] = (
            out["Close"].shift(1)
        )

        out["PREV_MACD"] = (
            out["MACD"].shift(1)
        )

        out["PREV_MACD_SIGNAL"] = (
            out["MACD_SIGNAL"].shift(1)
        )

        out["PREV_MACD_HIST"] = (
            out["MACD_HIST"].shift(1)
        )

        out["PREV_STOCH_K"] = (
            out["STOCH_K"].shift(1)
        )

        out["PREV_STOCH_D"] = (
            out["STOCH_D"].shift(1)
        )

        # ----------------------------------------------------
        # Daily trend state
        # ----------------------------------------------------

        out["TREND_BULL"] = (
            (out["Close"] > out["EMA5"])
            &
            (out["EMA5"] > out["EMA10"])
            &
            (out["EMA10"] > out["EMA75"])
        )

        # ----------------------------------------------------
        # Weekly trend state
        # ----------------------------------------------------

        out["WEEKLY_TREND_BULL"] = (
            out["W_EMA5"] > out["W_EMA10"]
        )

        # ----------------------------------------------------
        # Daily MACD bullish state
        # ----------------------------------------------------

        out["MACD_BULL"] = (
            out["MACD"] > out["MACD_SIGNAL"]
        )

        out["MACD_HIST_RISING"] = (
            out["MACD_HIST"]
            > out["PREV_MACD_HIST"]
        )

        # ----------------------------------------------------
        # Weekly MACD bullish state
        # ----------------------------------------------------

        out["WEEKLY_MACD_BULL"] = (
            out["W_MACD"] > out["W_MACD_SIGNAL"]
        )

        out["WEEKLY_MACD_HIST_RISING"] = (
            out["W_MACD_HIST"]
            > out["W_MACD_HIST"].shift(1)
        )

        # ----------------------------------------------------
        # Stochastic bullish state
        # ----------------------------------------------------

        out["STOCH_BULL"] = (
            (out["STOCH_K"] > out["STOCH_D"])
            &
            (out["STOCH_K"] < 80)
        )

        # ----------------------------------------------------
        # Volume confirmation
        # ----------------------------------------------------

        out["VOLUME_CONFIRM"] = (
            out["VOLUME_RATIO"]
            >= 1.2
        )

        # ----------------------------------------------------
        # Early-entry / squeeze state
        # ----------------------------------------------------

        out["EARLY_ENTRY_ZONE"] = (
            out["SQUEEZE"]
            |
            out["SQUEEZE_RELEASE"]
            |
            out["SQUEEZE_RECENT"]
        )

        # ----------------------------------------------------
        # Target prices
        # ----------------------------------------------------

        out["TARGET1"] = (
            out["Close"] * 1.07
        )

        out["TARGET2"] = (
            out["Close"] * 1.15
        )

        # ----------------------------------------------------
        # Final dataset
        # ----------------------------------------------------

        feature_data[symbol] = out

    except Exception as e:

        feature_errors[symbol] = str(e)


print("📦 Building feature datasets...\n")

for symbol, df in feature_data.items():

    print(
        f"✅ {symbol:<12} "
        f"{len(df):>5} usable bars"
    )


print("\n====================================")
print(
    f"Feature datasets: "
    f"{len(feature_data)} / {len(combined_data)}"
)
print("====================================")


if feature_errors:

    print("\n⚠️ Feature errors:")

    for symbol, error in feature_errors.items():

        print(
            f"{symbol}: {error}"
        )


# ============================================================
# VALIDATION
# ============================================================

if feature_data:

    sample_symbol = next(iter(feature_data))

    sample = feature_data[sample_symbol]

    print(
        f"\n🔎 Feature validation: {sample_symbol}"
    )

    display(
        sample[
            [
                "Close",
                "EMA5",
                "EMA10",
                "EMA75",
                "MACD",
                "MACD_SIGNAL",
                "MACD_HIST",
                "W_EMA5",
                "W_EMA10",
                "W_MACD",
                "W_MACD_SIGNAL",
                "W_MACD_HIST",
                "TREND_BULL",
                "WEEKLY_TREND_BULL",
                "MACD_BULL",
                "STOCH_BULL",
                "VOLUME_CONFIRM",
                "EARLY_ENTRY_ZONE"
            ]
        ].tail(10)
    )

📦 Building feature datasets...

✅ AU.BK         1206 usable bars
✅ CBG.BK        1206 usable bars
✅ DELTA.BK      1206 usable bars
✅ DITTO.BK      1206 usable bars
✅ FORTH.BK      1206 usable bars
✅ ICHI.BK       1206 usable bars
✅ JMART.BK      1206 usable bars
✅ JMT.BK        1206 usable bars
✅ KAMART.BK     1205 usable bars
✅ KLINIQ.BK      945 usable bars
✅ MASTER.BK      891 usable bars
✅ MEB.BK         877 usable bars
✅ MOSHI.BK       914 usable bars
✅ PLANB.BK      1206 usable bars
✅ SAPPE.BK      1206 usable bars
✅ SNNP.BK       1206 usable bars
✅ SPA.BK        1206 usable bars
✅ TKN.BK        1206 usable bars
✅ XO.BK         1206 usable bars

Feature datasets: 19 / 19

🔎 Feature validation: AU.BK


Price,Close,EMA5,EMA10,EMA75,MACD,MACD_SIGNAL,MACD_HIST,W_EMA5,W_EMA10,W_MACD,W_MACD_SIGNAL,W_MACD_HIST,TREND_BULL,WEEKLY_TREND_BULL,MACD_BULL,STOCH_BULL,VOLUME_CONFIRM,EARLY_ENTRY_ZONE
Date,,,,,,,,,,,,,,,,,,
2026-09-23,4.22,4.234850,4.246672,4.482507,-0.020207,-0.025758,0.005551,4.325448,4.425397,-0.153826,-0.103219,-0.050607,False,False,True,False,True,False
2026-09-24,4.22,4.229900,4.241822,4.475599,-0.020207,-0.024648,0.004441,4.325448,4.425397,-0.153826,-0.103219,-0.050607,False,False,True,False,True,False
2026-09-25,4.20,4.219933,4.234218,4.468346,-0.023887,-0.024496,0.000609,4.325448,4.425397,-0.153826,-0.103219,-0.050607,False,False,True,False,False,False
2026-09-28,4.20,4.213289,4.227997,4.461285,-0.024285,-0.024454,0.000169,4.283632,4.384416,-0.154994,-0.113574,-0.041420,False,False,True,True,True,True
2026-09-29,4.20,4.208859,4.222906,4.454409,-0.022993,-0.024162,0.001169,4.283632,4.384416,-0.154994,-0.113574,-0.041420,False,False,True,True,False,True
2026-09-30,4.16,4.192573,4.211469,4.446661,-0.030768,-0.025483,-0.005285,4.283632,4.384416,-0.154994,-0.113574,-0.041420,False,False,False,True,True,True
2026-10-01,4.14,4.175048,4.198474,4.438591,-0.037745,-0.027935,-0.009810,4.283632,4.384416,-0.154994,-0.113574,-0.041420,False,False,False,False,False,True
2026-10-02,4.14,4.163366,4.187843,4.430733,-0.038964,-0.030141,-0.008823,4.283632,4.384416,-0.154994,-0.113574,-0.041420,False,False,False,False,False,True
2026-10-05,4.20,4.175577,4.190053,4.424662,-0.022416,-0.028596,0.006180,4.235754,4.339977,-0.160818,-0.123023,-0.037795,False,False,True,True,True,True


In [48]:

# ============================================================
# 7) LEGACY SCORE
# ============================================================
#
# วัตถุประสงค์:
# - เก็บสูตร FAST HYBRID PRO เดิมไว้เป็น Baseline
# - ใช้เปรียบเทียบกับ Canonical Research Score ใน Cell 8
# - ยังไม่ทำ Backtest และยังไม่เปิด Position
#
# LEGACY 5-POINT SCORE
# 1. MACD 6/13/5 + Histogram Rising
# 2. Stochastic 8/3/3 : K > D และ K < 80
# 3. Trend: Close > EMA20 > EMA50
# 4. Candle Body / ATR >= 0.8
# 5. Volatility: BB breakout OR squeeze release
#
# หมายเหตุ:
# - Cell 4 ใช้ EMA5/10/75 และ MACD 4/12/9 เป็น canonical data
# - ดังนั้น Legacy MACD จะคำนวณใหม่ตรงนี้ด้วย 6/13/5
# - EMA20/EMA50 ก็จะคำนวณใหม่เพื่อรักษาสูตร Legacy เดิม
# ============================================================


def calculate_legacy_score(df):
    out = df.copy()

    # --------------------------------------------------------
    # 1) LEGACY MACD 6/13/5
    # --------------------------------------------------------
    close = out["Close"]

    legacy_ema_fast = close.ewm(
        span=6,
        adjust=False
    ).mean()

    legacy_ema_slow = close.ewm(
        span=13,
        adjust=False
    ).mean()

    out["LEGACY_MACD"] = (
        legacy_ema_fast - legacy_ema_slow
    )

    out["LEGACY_MACD_SIGNAL"] = (
        out["LEGACY_MACD"]
        .ewm(span=5, adjust=False)
        .mean()
    )

    out["LEGACY_MACD_HIST"] = (
        out["LEGACY_MACD"]
        - out["LEGACY_MACD_SIGNAL"]
    )

    out["LEGACY_MACD_HIST_PREV"] = (
        out["LEGACY_MACD_HIST"].shift(1)
    )

    out["LEGACY_MACD_BULL"] = (
        (out["LEGACY_MACD"] > out["LEGACY_MACD_SIGNAL"])
        &
        (
            out["LEGACY_MACD_HIST"]
            > out["LEGACY_MACD_HIST_PREV"]
        )
    )


    # --------------------------------------------------------
    # 2) LEGACY STOCHASTIC 8/3/3
    # --------------------------------------------------------
    lowest_8 = out["Low"].rolling(
        window=8,
        min_periods=8
    ).min()

    highest_8 = out["High"].rolling(
        window=8,
        min_periods=8
    ).max()

    price_range = (
        highest_8 - lowest_8
    ).replace(0, np.nan)

    stoch_raw = (
        100
        * (close - lowest_8)
        / price_range
    )

    out["LEGACY_STOCH_K"] = (
        stoch_raw
        .rolling(window=3, min_periods=3)
        .mean()
    )

    out["LEGACY_STOCH_D"] = (
        out["LEGACY_STOCH_K"]
        .rolling(window=3, min_periods=3)
        .mean()
    )

    out["LEGACY_STOCH_BULL"] = (
        (out["LEGACY_STOCH_K"] > out["LEGACY_STOCH_D"])
        &
        (out["LEGACY_STOCH_K"] < 80)
    )


    # --------------------------------------------------------
    # 3) LEGACY TREND
    # Close > EMA20 > EMA50
    # --------------------------------------------------------
    out["LEGACY_EMA20"] = (
        close.ewm(
            span=20,
            adjust=False
        ).mean()
    )

    out["LEGACY_EMA50"] = (
        close.ewm(
            span=50,
            adjust=False
        ).mean()
    )

    out["LEGACY_TREND_BULL"] = (
        (close > out["LEGACY_EMA20"])
        &
        (out["LEGACY_EMA20"] > out["LEGACY_EMA50"])
    )


    # --------------------------------------------------------
    # 4) CANDLE BODY / ATR >= 0.8
    # --------------------------------------------------------
    #
    # ใช้ ATR14 จาก Cell 4
    # Body = abs(Close - Open)
    #
    # เงื่อนไขเดิม:
    # Body / ATR >= 0.8
    # --------------------------------------------------------

    out["LEGACY_BODY_ATR_RATIO"] = (
        (out["Close"] - out["Open"]).abs()
        / out["ATR14"].replace(0, np.nan)
    )

    out["LEGACY_CANDLE_BULL"] = (
        out["LEGACY_BODY_ATR_RATIO"] >= 0.8
    )


    # --------------------------------------------------------
    # 5) VOLATILITY
    # BB Breakout OR Squeeze Release
    # --------------------------------------------------------
    #
    # BB breakout:
    # Close > Upper BB
    #
    # Squeeze release:
    # SQUEEZE วันนี้ False
    # และเมื่อวานเป็น True
    # --------------------------------------------------------

    out["LEGACY_BB_BREAKOUT"] = (
        out["Close"] > out["BB_UPPER"]
    )

    out["LEGACY_SQUEEZE_RELEASE"] = (
        (~out["SQUEEZE"])
        &
        (out["SQUEEZE"].shift(1).fillna(False))
    )

    out["LEGACY_VOLATILITY_BULL"] = (
        out["LEGACY_BB_BREAKOUT"]
        |
        out["LEGACY_SQUEEZE_RELEASE"]
    )


    # --------------------------------------------------------
    # LEGACY SCORE
    # --------------------------------------------------------

    score_components = [
        "LEGACY_MACD_BULL",
        "LEGACY_STOCH_BULL",
        "LEGACY_TREND_BULL",
        "LEGACY_CANDLE_BULL",
        "LEGACY_VOLATILITY_BULL",
    ]

    out["LEGACY_SCORE"] = (
        out[score_components]
        .fillna(False)
        .astype(int)
        .sum(axis=1)
    )


    # --------------------------------------------------------
    # Signal classification
    # --------------------------------------------------------
    #
    # Legacy เดิมใช้ 5 คะแนนเต็ม
    #
    # >= 3 = ผ่าน baseline
    # < 3  = ไม่ผ่าน
    # --------------------------------------------------------

    out["LEGACY_SIGNAL"] = (
        out["LEGACY_SCORE"] >= 3
    )


    return out


# ============================================================
# APPLY TO ALL SYMBOLS
# ============================================================

legacy_data = {}
legacy_errors = {}

print("🧮 Calculating Legacy Score...\n")

for symbol, df in feature_data.items():
    try:
        result = calculate_legacy_score(df)

        legacy_data[symbol] = result

        print(
            f"✅ {symbol:<12} "
            f"{len(result):>5} bars"
        )

    except Exception as e:
        legacy_errors[symbol] = str(e)

        print(
            f"❌ {symbol}: {e}"
        )


# ============================================================
# SUMMARY
# ============================================================

print("\n====================================")
print(
    f"Legacy datasets: "
    f"{len(legacy_data)} / {len(feature_data)}"
)
print("====================================")


if legacy_errors:
    print("\n⚠️ Legacy calculation errors:")

    for symbol, error in legacy_errors.items():
        print(f"{symbol}: {error}")


# ============================================================
# VALIDATION
# ============================================================

if legacy_data:

    sample_symbol = next(iter(legacy_data))
    sample = legacy_data[sample_symbol]

    print(
        f"\n🔎 Legacy validation: "
        f"{sample_symbol}"
    )

    display(
        sample[
            [
                "Close",
                "LEGACY_EMA20",
                "LEGACY_EMA50",

                "LEGACY_MACD",
                "LEGACY_MACD_SIGNAL",
                "LEGACY_MACD_HIST",

                "LEGACY_STOCH_K",
                "LEGACY_STOCH_D",

                "LEGACY_BODY_ATR_RATIO",

                "LEGACY_BB_BREAKOUT",
                "LEGACY_SQUEEZE_RELEASE",

                "LEGACY_MACD_BULL",
                "LEGACY_STOCH_BULL",
                "LEGACY_TREND_BULL",
                "LEGACY_CANDLE_BULL",
                "LEGACY_VOLATILITY_BULL",

                "LEGACY_SCORE",
                "LEGACY_SIGNAL",
            ]
        ].tail(10)
    )


# ============================================================
# SCORE DISTRIBUTION
# ============================================================

if legacy_data:

    all_scores = pd.concat(
        [
            df["LEGACY_SCORE"]
            for df in legacy_data.values()
        ]
    )

    print("\n📊 Legacy Score Distribution")

    display(
        all_scores
        .value_counts()
        .sort_index()
        .rename("bars")
        .to_frame()
    )

    print(
        f"\nBars with Legacy Score >= 3: "
        f"{int((all_scores >= 3).sum()):,}"
    )

    print(
        f"Total usable bars: "
        f"{len(all_scores):,}"
    )

    print(
        f"Signal rate: "
        f"{(all_scores >= 3).mean() * 100:.2f}%"
    )

🧮 Calculating Legacy Score...

✅ AU.BK         1206 bars
✅ CBG.BK        1206 bars
✅ DELTA.BK      1206 bars
✅ DITTO.BK      1206 bars
✅ FORTH.BK      1206 bars
✅ ICHI.BK       1206 bars
✅ JMART.BK      1206 bars
✅ JMT.BK        1206 bars
✅ KAMART.BK     1205 bars
✅ KLINIQ.BK      945 bars
✅ MASTER.BK      891 bars
✅ MEB.BK         877 bars
✅ MOSHI.BK       914 bars
✅ PLANB.BK      1206 bars
✅ SAPPE.BK      1206 bars
✅ SNNP.BK       1206 bars
✅ SPA.BK        1206 bars
✅ TKN.BK        1206 bars
✅ XO.BK         1206 bars

Legacy datasets: 19 / 19

🔎 Legacy validation: AU.BK


Price,Close,LEGACY_EMA20,LEGACY_EMA50,LEGACY_MACD,LEGACY_MACD_SIGNAL,LEGACY_MACD_HIST,LEGACY_STOCH_K,LEGACY_STOCH_D,LEGACY_BODY_ATR_RATIO,LEGACY_BB_BREAKOUT,LEGACY_SQUEEZE_RELEASE,LEGACY_MACD_BULL,LEGACY_STOCH_BULL,LEGACY_TREND_BULL,LEGACY_CANDLE_BULL,LEGACY_VOLATILITY_BULL,LEGACY_SCORE,LEGACY_SIGNAL
Date,,,,,,,,,,,,,,,,,,
2026-09-23,4.22,4.285613,4.410743,-0.019302,-0.021473,0.002171,22.222046,29.629394,0.441062,False,False,False,False,False,False,False,0,False
2026-09-24,4.22,4.279364,4.403263,-0.018971,-0.020639,0.001668,19.444306,24.999807,0.000000,False,False,False,False,False,False,False,0,False
2026-09-25,4.20,4.271805,4.395292,-0.020852,-0.020710,-0.000142,8.333284,16.666545,0.463176,False,False,False,False,False,False,False,0,False
2026-09-28,4.20,4.264967,4.387634,-0.021152,-0.020857,-0.000295,21.666553,16.481381,0.000000,False,False,False,True,False,False,False,1,False
2026-09-29,4.20,4.258780,4.380275,-0.020472,-0.020729,0.000257,26.666540,18.888792,0.469043,False,False,True,True,False,False,False,2,False
2026-09-30,4.16,4.249372,4.371637,-0.024935,-0.022131,-0.002804,26.666540,24.999877,0.942252,False,False,False,True,False,True,False,2,False
2026-10-01,4.14,4.238956,4.362553,-0.029506,-0.024589,-0.004917,21.666578,24.999886,0.416748,False,True,False,False,False,False,True,1,False
2026-10-02,4.14,4.229531,4.353826,-0.031101,-0.026760,-0.004341,16.666617,21.666578,0.000000,False,False,False,False,False,False,False,0,False
2026-10-05,4.20,4.226719,4.347793,-0.022236,-0.025252,0.003016,40.476141,26.269779,0.725226,False,False,True,True,False,False,False,2,False



📊 Legacy Score Distribution


,bars
LEGACY_SCORE,
0,6030
1,8130
2,5161
3,1665
4,578
5,152



Bars with Legacy Score >= 3: 2,395
Total usable bars: 21,716
Signal rate: 11.03%


In [49]:

# ============================================================
# 8) CANONICAL RESEARCH SCORE
# ============================================================
#
# PURPOSE
# ------------------------------------------------------------
# สร้างคะแนนระบบใหม่เพื่อใช้เปรียบเทียบกับ Legacy Score
#
# CANONICAL INDICATORS
#   DAY MACD      = 4 / 12 / 9
#   STOCHASTIC    = 8 / 3 / 3
#   DAY EMA       = 5 / 10 / 75
#   WEEK EMA      = 5 / 10
#   WEEK MACD     = 4 / 12 / 9
#
# RESEARCH SCORE = 5 POINTS
#
# 1) MACD
#    DAY MACD > Signal
#    AND Histogram > 0
#
# 2) STOCHASTIC
#    K > D
#    AND K < 80
#
# 3) DAY TREND
#    Close > EMA5 > EMA10 > EMA75
#
# 4) EARLY ENTRY
#    Squeeze OR Squeeze Release
#
# 5) VOLUME
#    Volume Ratio >= 1.2
#
# PASS:
#    Score >= 3
#
# IMPORTANT
# ------------------------------------------------------------
# Cell นี้เป็น Research Baseline
# ไม่ใช่ Production Weighted Score
# ============================================================


def calculate_canonical_research_score(df):

    out = df.copy()

    # --------------------------------------------------------
    # 1) DAY MACD 4/12/9
    # --------------------------------------------------------

    out["CANONICAL_MACD_BULL"] = (
        (out["MACD"] > out["MACD_SIGNAL"])
        &
        (out["MACD_HIST"] > 0)
    )


    # --------------------------------------------------------
    # 2) STOCHASTIC 8/3/3
    # --------------------------------------------------------

    out["CANONICAL_STOCH_BULL"] = (
        (out["STOCH_K"] > out["STOCH_D"])
        &
        (out["STOCH_K"] < 80)
    )


    # --------------------------------------------------------
    # 3) DAY TREND
    #
    # Close > EMA5 > EMA10 > EMA75
    # --------------------------------------------------------

    out["CANONICAL_TREND_BULL"] = (
        (out["Close"] > out["EMA5"])
        &
        (out["EMA5"] > out["EMA10"])
        &
        (out["EMA10"] > out["EMA75"])
    )


    # --------------------------------------------------------
    # 4) EARLY ENTRY
    #
    # ใช้ Squeeze หรือ Squeeze Release
    #
    # ไม่ใช้ SQUEEZE_RECENT ในคะแนนหลัก
    # เพื่อไม่ให้ signal window กว้างเกินไป
    # --------------------------------------------------------

    out["CANONICAL_EARLY_ENTRY"] = (
        out["SQUEEZE"]
        |
        out["SQUEEZE_RELEASE"]
    )


    # --------------------------------------------------------
    # 5) VOLUME CONFIRMATION
    # --------------------------------------------------------

    out["CANONICAL_VOLUME_CONFIRM"] = (
        out["VOLUME_RATIO"] >= 1.2
    )


    # --------------------------------------------------------
    # SCORE COMPONENTS
    # --------------------------------------------------------

    canonical_components = [
        "CANONICAL_MACD_BULL",
        "CANONICAL_STOCH_BULL",
        "CANONICAL_TREND_BULL",
        "CANONICAL_EARLY_ENTRY",
        "CANONICAL_VOLUME_CONFIRM",
    ]

    out["CANONICAL_SCORE"] = (
        out[canonical_components]
        .fillna(False)
        .astype(int)
        .sum(axis=1)
    )


    # --------------------------------------------------------
    # SIGNAL
    # --------------------------------------------------------

    out["CANONICAL_SIGNAL"] = (
        out["CANONICAL_SCORE"] >= 3
    )


    # --------------------------------------------------------
    # ADDITIONAL INFORMATION
    # --------------------------------------------------------
    #
    # Weekly confirmation ไม่ถูกนับเป็นคะแนนเพิ่ม
    # ใน Research Score
    #
    # เก็บไว้เพื่อใช้วิเคราะห์ภายหลัง
    # --------------------------------------------------------

    out["CANONICAL_WEEKLY_TREND"] = (
        out["W_EMA5"] > out["W_EMA10"]
    )

    out["CANONICAL_WEEKLY_MACD"] = (
        out["W_MACD"] > out["W_MACD_SIGNAL"]
    )

    out["CANONICAL_WEEKLY_CONFIRM"] = (
        out["CANONICAL_WEEKLY_TREND"]
        &
        out["CANONICAL_WEEKLY_MACD"]
    )


    return out


# ============================================================
# APPLY TO ALL SYMBOLS
# ============================================================

canonical_data = {}
canonical_errors = {}

print("🧮 Calculating Canonical Research Score...\n")

for symbol, df in feature_data.items():

    try:

        result = calculate_canonical_research_score(df)

        canonical_data[symbol] = result

        print(
            f"✅ {symbol:<12} "
            f"{len(result):>5} bars"
        )

    except Exception as e:

        canonical_errors[symbol] = str(e)

        print(
            f"❌ {symbol}: {e}"
        )


# ============================================================
# SUMMARY
# ============================================================

print("\n====================================")
print(
    f"Canonical datasets: "
    f"{len(canonical_data)} / {len(feature_data)}"
)
print("====================================")


if canonical_errors:

    print("\n⚠️ Canonical calculation errors:")

    for symbol, error in canonical_errors.items():

        print(
            f"{symbol}: {error}"
        )


# ============================================================
# VALIDATION
# ============================================================

if canonical_data:

    sample_symbol = next(iter(canonical_data))

    sample = canonical_data[sample_symbol]

    print(
        f"\n🔎 Canonical validation: "
        f"{sample_symbol}"
    )

    display(
        sample[
            [
                "Close",

                "EMA5",
                "EMA10",
                "EMA75",

                "MACD",
                "MACD_SIGNAL",
                "MACD_HIST",

                "STOCH_K",
                "STOCH_D",

                "W_EMA5",
                "W_EMA10",

                "W_MACD",
                "W_MACD_SIGNAL",
                "W_MACD_HIST",

                "VOLUME_RATIO",

                "SQUEEZE",
                "SQUEEZE_RELEASE",

                "CANONICAL_MACD_BULL",
                "CANONICAL_STOCH_BULL",
                "CANONICAL_TREND_BULL",
                "CANONICAL_EARLY_ENTRY",
                "CANONICAL_VOLUME_CONFIRM",

                "CANONICAL_SCORE",
                "CANONICAL_SIGNAL",

                "CANONICAL_WEEKLY_TREND",
                "CANONICAL_WEEKLY_MACD",
                "CANONICAL_WEEKLY_CONFIRM",
            ]
        ].tail(10)
    )


# ============================================================
# SCORE DISTRIBUTION
# ============================================================

if canonical_data:

    all_canonical_scores = pd.concat(
        [
            df["CANONICAL_SCORE"]
            for df in canonical_data.values()
        ]
    )

    print("\n📊 Canonical Score Distribution")

    display(
        all_canonical_scores
        .value_counts()
        .sort_index()
        .rename("bars")
        .to_frame()
    )


    signal_count = int(
        (all_canonical_scores >= 3).sum()
    )

    total_bars = len(
        all_canonical_scores
    )

    signal_rate = (
        signal_count / total_bars * 100
        if total_bars > 0
        else 0
    )


    print(
        f"\nBars with Canonical Score >= 3: "
        f"{signal_count:,}"
    )

    print(
        f"Total usable bars: "
        f"{total_bars:,}"
    )

    print(
        f"Canonical signal rate: "
        f"{signal_rate:.2f}%"
    )


# ============================================================
# LEGACY vs CANONICAL QUICK COMPARISON
# ============================================================

if legacy_data and canonical_data:

    legacy_scores = pd.concat(
        [
            df["LEGACY_SCORE"]
            for df in legacy_data.values()
        ]
    )

    canonical_scores = pd.concat(
        [
            df["CANONICAL_SCORE"]
            for df in canonical_data.values()
        ]
    )

    print("\n====================================")
    print("LEGACY vs CANONICAL")
    print("====================================")

    print(
        f"Legacy signal rate:     "
        f"{(legacy_scores >= 3).mean() * 100:.2f}%"
    )

    print(
        f"Canonical signal rate:  "
        f"{(canonical_scores >= 3).mean() * 100:.2f}%"
    )

🧮 Calculating Canonical Research Score...

✅ AU.BK         1206 bars
✅ CBG.BK        1206 bars
✅ DELTA.BK      1206 bars
✅ DITTO.BK      1206 bars
✅ FORTH.BK      1206 bars
✅ ICHI.BK       1206 bars
✅ JMART.BK      1206 bars
✅ JMT.BK        1206 bars
✅ KAMART.BK     1205 bars
✅ KLINIQ.BK      945 bars
✅ MASTER.BK      891 bars
✅ MEB.BK         877 bars
✅ MOSHI.BK       914 bars
✅ PLANB.BK      1206 bars
✅ SAPPE.BK      1206 bars
✅ SNNP.BK       1206 bars
✅ SPA.BK        1206 bars
✅ TKN.BK        1206 bars
✅ XO.BK         1206 bars

Canonical datasets: 19 / 19

🔎 Canonical validation: AU.BK


Price,Close,EMA5,EMA10,EMA75,MACD,MACD_SIGNAL,MACD_HIST,STOCH_K,STOCH_D,W_EMA5,W_EMA10,W_MACD,W_MACD_SIGNAL,W_MACD_HIST,VOLUME_RATIO,SQUEEZE,SQUEEZE_RELEASE,CANONICAL_MACD_BULL,CANONICAL_STOCH_BULL,CANONICAL_TREND_BULL,CANONICAL_EARLY_ENTRY,CANONICAL_VOLUME_CONFIRM,CANONICAL_SCORE,CANONICAL_SIGNAL,CANONICAL_WEEKLY_TREND,CANONICAL_WEEKLY_MACD,CANONICAL_WEEKLY_CONFIRM
Date,,,,,,,,,,,,,,,,,,,,,,,,,,,
2026-09-23,4.22,4.234850,4.246672,4.482507,-0.020207,-0.025758,0.005551,22.222046,29.629394,4.325448,4.425397,-0.153826,-0.103219,-0.050607,1.816975,False,False,True,False,False,False,True,2,False,False,False,False
2026-09-24,4.22,4.229900,4.241822,4.475599,-0.020207,-0.024648,0.004441,19.444306,24.999807,4.325448,4.425397,-0.153826,-0.103219,-0.050607,1.484927,False,False,True,False,False,False,True,2,False,False,False,False
2026-09-25,4.20,4.219933,4.234218,4.468346,-0.023887,-0.024496,0.000609,8.333284,16.666545,4.325448,4.425397,-0.153826,-0.103219,-0.050607,0.959982,False,False,True,False,False,False,False,1,False,False,False,False
2026-09-28,4.20,4.213289,4.227997,4.461285,-0.024285,-0.024454,0.000169,21.666553,16.481381,4.283632,4.384416,-0.154994,-0.113574,-0.041420,1.321540,True,False,True,True,False,True,True,4,True,False,False,False
2026-09-29,4.20,4.208859,4.222906,4.454409,-0.022993,-0.024162,0.001169,26.666540,18.888792,4.283632,4.384416,-0.154994,-0.113574,-0.041420,0.534683,True,False,True,True,False,True,False,3,True,False,False,False
2026-09-30,4.16,4.192573,4.211469,4.446661,-0.030768,-0.025483,-0.005285,26.666540,24.999877,4.283632,4.384416,-0.154994,-0.113574,-0.041420,1.524227,True,False,False,True,False,True,True,3,True,False,False,False
2026-10-01,4.14,4.175048,4.198474,4.438591,-0.037745,-0.027935,-0.009810,21.666578,24.999886,4.283632,4.384416,-0.154994,-0.113574,-0.041420,1.063792,False,True,False,False,False,True,False,1,False,False,False,False
2026-10-02,4.14,4.163366,4.187843,4.430733,-0.038964,-0.030141,-0.008823,16.666617,21.666578,4.283632,4.384416,-0.154994,-0.113574,-0.041420,1.187989,False,False,False,False,False,False,False,0,False,False,False,False
2026-10-05,4.20,4.175577,4.190053,4.424662,-0.022416,-0.028596,0.006180,40.476141,26.269779,4.235754,4.339977,-0.160818,-0.123023,-0.037795,1.358750,True,False,True,True,False,True,True,4,True,False,False,False



📊 Canonical Score Distribution


,bars
CANONICAL_SCORE,
0,3584
1,7202
2,6693
3,3027
4,987
5,223



Bars with Canonical Score >= 3: 4,237
Total usable bars: 21,716
Canonical signal rate: 19.51%

LEGACY vs CANONICAL
Legacy signal rate:     11.03%
Canonical signal rate:  19.51%


In [50]:

# ============================================================
# 9) MONEY MANAGEMENT — FINAL
# ============================================================

INITIAL_CAPITAL = 100_000.0
MAX_RISK_PCT = 0.02
RISK_PER_TRADE = INITIAL_CAPITAL * MAX_RISK_PCT

LOT_SIZE = 100
ATR_STOP_MULT = 1.5
RECENT_LOW_LOOKBACK = 5


print("💰 Money Management Configuration")
print("------------------------------------")
print(f"Initial Capital : {INITIAL_CAPITAL:,.2f} THB")
print(f"Risk / Trade    : {MAX_RISK_PCT * 100:.2f}%")
print(f"Risk Amount     : {RISK_PER_TRADE:,.2f} THB")
print(f"Lot Size        : {LOT_SIZE:,} shares")
print(f"ATR Stop Mult   : {ATR_STOP_MULT:.2f}x")
print(f"Recent Low      : {RECENT_LOW_LOOKBACK} bars")


# ============================================================
# STOP PRICE
# ============================================================

def calculate_stop_price(
    entry_price,
    atr,
    recent_low,
    atr_multiplier=ATR_STOP_MULT,
):
    if pd.isna(entry_price) or entry_price <= 0:
        return np.nan

    if pd.isna(atr) or atr <= 0:
        return np.nan

    if pd.isna(recent_low) or recent_low <= 0:
        return np.nan

    atr_stop = entry_price - atr * atr_multiplier

    # Conservative stop:
    # choose the LOWER of ATR stop and recent low
    stop_price = min(atr_stop, recent_low)

    if stop_price <= 0:
        return np.nan

    if stop_price >= entry_price:
        return np.nan

    return float(stop_price)


# ============================================================
# POSITION SIZE
# ============================================================

def calculate_position_size(
    capital,
    entry_price,
    stop_price,
    risk_pct=MAX_RISK_PCT,
    lot_size=LOT_SIZE,
):
    if pd.isna(capital) or capital <= 0:
        return {
            "shares": 0,
            "lots": 0,
            "risk_amount": 0.0,
            "risk_per_share": np.nan,
            "position_value": 0.0,
            "status": "INVALID_CAPITAL",
        }

    if pd.isna(entry_price) or entry_price <= 0:
        return {
            "shares": 0,
            "lots": 0,
            "risk_amount": 0.0,
            "risk_per_share": np.nan,
            "position_value": 0.0,
            "status": "INVALID_ENTRY",
        }

    if pd.isna(stop_price) or stop_price <= 0:
        return {
            "shares": 0,
            "lots": 0,
            "risk_amount": 0.0,
            "risk_per_share": np.nan,
            "position_value": 0.0,
            "status": "INVALID_STOP",
        }

    if stop_price >= entry_price:
        return {
            "shares": 0,
            "lots": 0,
            "risk_amount": 0.0,
            "risk_per_share": np.nan,
            "position_value": 0.0,
            "status": "STOP_NOT_BELOW_ENTRY",
        }

    max_risk = capital * risk_pct
    risk_per_share = entry_price - stop_price

    if risk_per_share <= 0:
        return {
            "shares": 0,
            "lots": 0,
            "risk_amount": 0.0,
            "risk_per_share": float(risk_per_share),
            "position_value": 0.0,
            "status": "INVALID_RISK_PER_SHARE",
        }

    # --------------------------------------------------------
    # Limit #1: risk-based position size
    # --------------------------------------------------------

    risk_lots = int(
        (max_risk / risk_per_share) // lot_size
    )

    # --------------------------------------------------------
    # Limit #2: capital-based position size
    # --------------------------------------------------------

    capital_lots = int(
        (capital / entry_price) // lot_size
    )

    # Final lots must satisfy BOTH constraints
    lots = min(risk_lots, capital_lots)

    shares = lots * lot_size

    if shares < lot_size:
        if capital_lots < 1:
            status = "INSUFFICIENT_CAPITAL_FOR_ONE_LOT"
        else:
            status = "ONE_LOT_EXCEEDS_RISK"

        return {
            "shares": 0,
            "lots": 0,
            "risk_amount": 0.0,
            "risk_per_share": float(risk_per_share),
            "position_value": 0.0,
            "status": status,
        }

    actual_risk = shares * risk_per_share
    position_value = shares * entry_price

    if capital_lots < risk_lots:
        status = "CAPITAL_LIMITED"
    else:
        status = "OK"

    return {
        "shares": int(shares),
        "lots": int(lots),
        "risk_amount": float(actual_risk),
        "risk_per_share": float(risk_per_share),
        "position_value": float(position_value),
        "status": status,
    }


# ============================================================
# COMPLETE TRADE PLAN
# ============================================================

def build_trade_plan(
    capital,
    entry_price,
    atr,
    recent_low,
    risk_pct=MAX_RISK_PCT,
    atr_multiplier=ATR_STOP_MULT,
    lot_size=LOT_SIZE,
):
    stop_price = calculate_stop_price(
        entry_price,
        atr,
        recent_low,
        atr_multiplier,
    )

    if pd.isna(stop_price):
        return {
            "entry_price": (
                float(entry_price)
                if pd.notna(entry_price)
                else np.nan
            ),
            "stop_price": np.nan,
            "shares": 0,
            "lots": 0,
            "risk_per_share": np.nan,
            "risk_amount": 0.0,
            "position_value": 0.0,
            "status": "INVALID_STOP",
        }

    position = calculate_position_size(
        capital,
        entry_price,
        stop_price,
        risk_pct,
        lot_size,
    )

    return {
        "entry_price": float(entry_price),
        "stop_price": float(stop_price),
        "shares": position["shares"],
        "lots": position["lots"],
        "risk_per_share": position["risk_per_share"],
        "risk_amount": position["risk_amount"],
        "position_value": position["position_value"],
        "status": position["status"],
    }


# ============================================================
# RECENT LOW
# ============================================================

for symbol, df in canonical_data.items():
    df["RECENT_LOW_5"] = (
        df["Low"]
        .rolling(
            window=RECENT_LOW_LOOKBACK,
            min_periods=RECENT_LOW_LOOKBACK,
        )
        .min()
    )


# ============================================================
# VALIDATION
# ============================================================

print("\n====================================")
print("🧪 Money Management Validation")
print("====================================")


# ------------------------------------------------------------
# Test 1: Normal trade
# ------------------------------------------------------------

test_entry = 50.00
test_atr = 1.50
test_recent_low = 47.00

test_plan = build_trade_plan(
    INITIAL_CAPITAL,
    test_entry,
    test_atr,
    test_recent_low,
)

print("\nTest 1: Normal Trade")
print("------------------------------------")

for key, value in test_plan.items():
    if isinstance(value, float):
        print(f"{key:<20}: {value:,.4f}")
    else:
        print(f"{key:<20}: {value}")


# ------------------------------------------------------------
# Test 2: Stop validation
# ------------------------------------------------------------

test_entry = 10.00
test_atr = 0.50
test_recent_low = 9.80

test_plan_2 = build_trade_plan(
    INITIAL_CAPITAL,
    test_entry,
    test_atr,
    test_recent_low,
)

print("\nTest 2: Stop Validation")
print("------------------------------------")
print(f"Entry : {test_plan_2['entry_price']:.4f}")
print(f"Stop  : {test_plan_2['stop_price']:.4f}")

if pd.notna(test_plan_2["stop_price"]):
    print(
        "Valid : "
        f"{test_plan_2['stop_price'] < test_plan_2['entry_price']}"
    )
else:
    print("Valid : False")


# ------------------------------------------------------------
# Test 3: One lot risk check
# ------------------------------------------------------------

test_entry = 80.00
test_stop = 60.00

test_position = calculate_position_size(
    INITIAL_CAPITAL,
    test_entry,
    test_stop,
)

print("\nTest 3: One Lot Risk Check")
print("------------------------------------")
print(f"Entry       : {test_entry:.2f}")
print(f"Stop        : {test_stop:.2f}")
print(f"Risk / Share: {test_position['risk_per_share']:.2f}")
print(f"Shares      : {test_position['shares']:,}")
print(f"Lots        : {test_position['lots']:,}")
print(f"Risk Amount : {test_position['risk_amount']:,.2f}")
print(f"Position    : {test_position['position_value']:,.2f}")
print(f"Status      : {test_position['status']}")


# ------------------------------------------------------------
# Test 4: Capital limit
# ------------------------------------------------------------

test_entry = 1.00
test_stop = 0.99

test_position_4 = calculate_position_size(
    INITIAL_CAPITAL,
    test_entry,
    test_stop,
)

print("\nTest 4: Capital Limit Check")
print("------------------------------------")
print(f"Entry       : {test_entry:.2f}")
print(f"Stop        : {test_stop:.2f}")
print(f"Risk / Share: {test_position_4['risk_per_share']:.4f}")
print(f"Shares      : {test_position_4['shares']:,}")
print(f"Lots        : {test_position_4['lots']:,}")
print(f"Risk Amount : {test_position_4['risk_amount']:,.2f}")
print(f"Position    : {test_position_4['position_value']:,.2f}")
print(f"Status      : {test_position_4['status']}")

assert (
    test_position_4["position_value"]
    <= INITIAL_CAPITAL + 1e-9
), "Position value exceeds capital"


# ============================================================
# REAL DATA SANITY CHECK
# ============================================================

print("\n====================================")
print("🔎 Real Data Sanity Check")
print("====================================")

if canonical_data:
    sample_symbol = next(iter(canonical_data))
    sample = canonical_data[sample_symbol]

    valid_rows = (
        sample[
            ["Close", "ATR14", "RECENT_LOW_5"]
        ]
        .dropna()
        .tail(3)
    )

    print(f"\nSample: {sample_symbol}")
    display(valid_rows)


# ============================================================
# FINAL CHECKS
# ============================================================

assert INITIAL_CAPITAL > 0
assert 0 < MAX_RISK_PCT <= 0.05
assert LOT_SIZE > 0
assert ATR_STOP_MULT > 0
assert RECENT_LOW_LOOKBACK > 0

print("\n====================================")
print("✅ Money Management Cell Ready")
print("====================================")
print(f"Maximum risk per trade: {RISK_PER_TRADE:,.2f} THB")
print("Stop rule: min(Entry - ATR × 1.5, Recent Low)")
print("Position size: whole 100-share lots, rounded DOWN")
print("Position limit: Risk limit + Capital limit")
print("Target prices: calculated later from actual Entry")

💰 Money Management Configuration
------------------------------------
Initial Capital : 100,000.00 THB
Risk / Trade    : 2.00%
Risk Amount     : 2,000.00 THB
Lot Size        : 100 shares
ATR Stop Mult   : 1.50x
Recent Low      : 5 bars

🧪 Money Management Validation

Test 1: Normal Trade
------------------------------------
entry_price         : 50.0000
stop_price          : 47.0000
shares              : 600
lots                : 6
risk_per_share      : 3.0000
risk_amount         : 1,800.0000
position_value      : 30,000.0000
status              : OK

Test 2: Stop Validation
------------------------------------
Entry : 10.0000
Stop  : 9.2500
Valid : True

Test 3: One Lot Risk Check
------------------------------------
Entry       : 80.00
Stop        : 60.00
Risk / Share: 20.00
Shares      : 100
Lots        : 1
Risk Amount : 2,000.00
Position    : 8,000.00
Status      : OK

Test 4: Capital Limit Check
------------------------------------
Entry       : 1.00
Stop        : 0.99
Risk / Shar

Price,Close,ATR14,RECENT_LOW_5
Date,,,
2026-10-02,4.14,0.051706,4.1
2026-10-05,4.20,0.055155,4.1
2026-10-06,4.18,0.054073,4.1



✅ Money Management Cell Ready
Maximum risk per trade: 2,000.00 THB
Stop rule: min(Entry - ATR × 1.5, Recent Low)
Position size: whole 100-share lots, rounded DOWN
Position limit: Risk limit + Capital limit
Target prices: calculated later from actual Entry


In [51]:

# ============================================================
# 10) CURRENT SCANNER
# ============================================================

print("🔎 CURRENT SCANNER")
print("=" * 90)


def build_current_scan_row(symbol, df):
    if df is None or df.empty:
        return {
            "Symbol": symbol,
            "Date": pd.NaT,
            "Close": np.nan,
            "Score": np.nan,
            "Signal": False,
            "WeeklyConfirm": False,
            "MACD": np.nan,
            "MACD_Signal": np.nan,
            "MACD_Hist": np.nan,
            "EMA5": np.nan,
            "EMA10": np.nan,
            "EMA75": np.nan,
            "W_EMA5": np.nan,
            "W_EMA10": np.nan,
            "W_MACD": np.nan,
            "W_MACD_Signal": np.nan,
            "ATR14": np.nan,
            "RecentLow5": np.nan,
            "Stop": np.nan,
            "RiskPerShare": np.nan,
            "Shares": 0,
            "Lots": 0,
            "RiskAmount": 0.0,
            "PositionValue": 0.0,
            "Target1": np.nan,
            "Target2": np.nan,
            "Status": "NO_DATA",
        }

    row = df.iloc[-1]

    entry_price = row["Close"]
    atr = row["ATR14"]
    recent_low = row["RECENT_LOW_5"]

    trade_plan = build_trade_plan(
        INITIAL_CAPITAL,
        entry_price,
        atr,
        recent_low,
    )

    score = row.get("CANONICAL_SCORE", np.nan)
    signal = bool(row.get("CANONICAL_SIGNAL", False))

    weekly_trend = bool(
        row.get("CANONICAL_WEEKLY_TREND", False)
    )

    weekly_macd = bool(
        row.get("CANONICAL_WEEKLY_MACD", False)
    )

    weekly_confirm = bool(
        row.get("CANONICAL_WEEKLY_CONFIRM", False)
    )

    target1 = (
        float(entry_price * 1.07)
        if pd.notna(entry_price)
        else np.nan
    )

    target2 = (
        float(entry_price * 1.15)
        if pd.notna(entry_price)
        else np.nan
    )

    return {
        "Symbol": symbol,
        "Date": df.index[-1],
        "Close": float(entry_price),
        "Score": score,
        "Signal": signal,
        "WeeklyTrend": weekly_trend,
        "WeeklyMACD": weekly_macd,
        "WeeklyConfirm": weekly_confirm,
        "MACD": row["MACD"],
        "MACD_Signal": row["MACD_SIGNAL"],
        "MACD_Hist": row["MACD_HIST"],
        "EMA5": row["EMA5"],
        "EMA10": row["EMA10"],
        "EMA75": row["EMA75"],
        "W_EMA5": row["W_EMA5"],
        "W_EMA10": row["W_EMA10"],
        "W_MACD": row["W_MACD"],
        "W_MACD_Signal": row["W_MACD_SIGNAL"],
        "ATR14": row["ATR14"],
        "RecentLow5": row["RECENT_LOW_5"],
        "Stop": trade_plan["stop_price"],
        "RiskPerShare": trade_plan["risk_per_share"],
        "Shares": trade_plan["shares"],
        "Lots": trade_plan["lots"],
        "RiskAmount": trade_plan["risk_amount"],
        "PositionValue": trade_plan["position_value"],
        "Target1": target1,
        "Target2": target2,
        "Status": trade_plan["status"],
    }


# ============================================================
# BUILD CURRENT SCAN
# ============================================================

current_scan_rows = []

for symbol, df in canonical_data.items():
    current_scan_rows.append(
        build_current_scan_row(symbol, df)
    )

current_scan = pd.DataFrame(current_scan_rows)


# ============================================================
# SORT
# ============================================================

current_scan = current_scan.sort_values(
    by=["Signal", "Score", "WeeklyConfirm"],
    ascending=[False, False, False],
    na_position="last",
).reset_index(drop=True)


# ============================================================
# DISPLAY
# ============================================================

print("\n📊 Current Market Snapshot")
print("-" * 90)

display(
    current_scan[
        [
            "Symbol",
            "Date",
            "Close",
            "Score",
            "Signal",
            "WeeklyConfirm",
            "ATR14",
            "Stop",
            "Shares",
            "RiskAmount",
            "Target1",
            "Target2",
            "Status",
        ]
    ]
)


# ============================================================
# SIGNAL SUMMARY
# ============================================================

signal_scan = current_scan[
    current_scan["Signal"] == True
].copy()

weekly_confirmed = current_scan[
    current_scan["WeeklyConfirm"] == True
].copy()

print("\n====================================")
print("📈 CURRENT SCANNER SUMMARY")
print("====================================")

print(f"Total symbols       : {len(current_scan)}")
print(f"Canonical signals   : {len(signal_scan)}")
print(f"Weekly confirmed    : {len(weekly_confirmed)}")

if len(current_scan) > 0:
    signal_rate = (
        len(signal_scan) / len(current_scan) * 100
    )
else:
    signal_rate = 0.0

print(f"Signal rate         : {signal_rate:.2f}%")


# ============================================================
# TOP SIGNALS
# ============================================================

print("\n====================================")
print("🏆 TOP CURRENT SIGNALS")
print("====================================")

if signal_scan.empty:
    print("No current Canonical signals.")
else:
    display(
        signal_scan[
            [
                "Symbol",
                "Date",
                "Close",
                "Score",
                "WeeklyConfirm",
                "ATR14",
                "Stop",
                "Shares",
                "RiskAmount",
                "Target1",
                "Target2",
            ]
        ].head(10)
    )


# ============================================================
# SANITY CHECKS
# ============================================================

assert len(current_scan) == len(canonical_data)

assert current_scan["Score"].dropna().between(0, 5).all()

assert (
    current_scan["RiskAmount"].fillna(0)
    <= RISK_PER_TRADE + 1e-9
).all()

assert (
    current_scan["PositionValue"].fillna(0)
    <= INITIAL_CAPITAL + 1e-9
).all()

print("\n====================================")
print("✅ Current Scanner Cell Ready")
print("====================================")

🔎 CURRENT SCANNER

📊 Current Market Snapshot
------------------------------------------------------------------------------------------


,Symbol,Date,Close,Score,Signal,WeeklyConfirm,ATR14,Stop,Shares,RiskAmount,Target1,Target2,Status
0,AU.BK,2026-10-06,4.18,3,True,False,0.054073,4.098891,23900,1938.504036,4.4726,4.8070,CAPITAL_LIMITED
1,DELTA.BK,2026-10-06,272.00,3,True,False,9.292522,252.000000,100,2000.000000,291.0400,312.8000,OK
2,SNNP.BK,2026-10-06,6.70,3,True,False,0.100529,6.500000,10000,1999.998093,7.1690,7.7050,OK
3,XO.BK,2026-10-06,20.00,3,True,False,0.535992,17.500000,800,2000.000000,21.4000,23.0000,OK
4,PLANB.BK,2026-10-06,7.80,2,False,True,0.331470,6.600000,1600,1920.000458,8.3460,8.9700,OK
5,CBG.BK,2026-10-06,53.25,2,False,False,1.685154,50.000000,600,1950.000000,56.9775,61.2375,OK
6,FORTH.BK,2026-10-06,13.90,2,False,False,0.717923,12.823115,1800,1938.392502,14.8730,15.9850,OK
7,JMART.BK,2026-10-06,9.70,2,False,False,0.269137,9.296295,4900,1978.154554,10.3790,11.1550,OK
8,KAMART.BK,2026-10-06,7.15,2,False,False,0.170746,6.650000,4000,2000.000000,7.6505,8.2225,OK
9,KLINIQ.BK,2026-10-06,28.25,2,False,False,0.719718,27.170422,1800,1943.239707,30.2275,32.4875,OK



📈 CURRENT SCANNER SUMMARY
Total symbols       : 19
Canonical signals   : 4
Weekly confirmed    : 2
Signal rate         : 21.05%

🏆 TOP CURRENT SIGNALS


,Symbol,Date,Close,Score,WeeklyConfirm,ATR14,Stop,Shares,RiskAmount,Target1,Target2
0,AU.BK,2026-10-06,4.18,3,False,0.054073,4.098891,23900,1938.504036,4.4726,4.807
1,DELTA.BK,2026-10-06,272.00,3,False,9.292522,252.000000,100,2000.000000,291.0400,312.800
2,SNNP.BK,2026-10-06,6.70,3,False,0.100529,6.500000,10000,1999.998093,7.1690,7.705
3,XO.BK,2026-10-06,20.00,3,False,0.535992,17.500000,800,2000.000000,21.4000,23.000



✅ Current Scanner Cell Ready


In [52]:

# ============================================================
# 11) HISTORICAL SIGNAL GENERATION
# ============================================================

print("📡 HISTORICAL SIGNAL GENERATION")
print("=" * 90)


SIGNAL_MIN_SCORE = 3


def generate_historical_signals(
    symbol,
    df,
    min_score=SIGNAL_MIN_SCORE,
):
    if df is None or df.empty:
        return pd.DataFrame()

    data = df.copy()

    # --------------------------------------------------------
    # Required columns
    # --------------------------------------------------------

    required = [
        "Close",
        "Open",
        "High",
        "Low",
        "ATR14",
        "RECENT_LOW_5",
        "CANONICAL_SCORE",
        "CANONICAL_SIGNAL",
        "CANONICAL_WEEKLY_TREND",
        "CANONICAL_WEEKLY_MACD",
        "CANONICAL_WEEKLY_CONFIRM",
    ]

    missing = [
        col for col in required
        if col not in data.columns
    ]

    if missing:
        raise ValueError(
            f"{symbol}: missing columns {missing}"
        )

    # --------------------------------------------------------
    # Signal rows
    # --------------------------------------------------------

    signal_mask = (
        data["CANONICAL_SIGNAL"].fillna(False)
        & (data["CANONICAL_SCORE"] >= min_score)
    )

    signal_dates = data.index[signal_mask]

    rows = []

    for signal_date in signal_dates:
        signal_pos = data.index.get_loc(signal_date)

        # Need next trading day for entry
        if signal_pos + 1 >= len(data):
            continue

        entry_date = data.index[signal_pos + 1]

        signal_row = data.iloc[signal_pos]
        entry_row = data.iloc[signal_pos + 1]

        signal_close = signal_row["Close"]
        signal_atr = signal_row["ATR14"]
        recent_low = signal_row["RECENT_LOW_5"]

        # ----------------------------------------------------
        # Stop is calculated from information available
        # at SIGNAL DATE.
        # ----------------------------------------------------

        stop_price = calculate_stop_price(
            signal_close,
            signal_atr,
            recent_low,
        )

        # ----------------------------------------------------
        # Entry is NEXT DAY OPEN.
        # ----------------------------------------------------

        entry_price = entry_row["Open"]

        # ----------------------------------------------------
        # If entry is invalid, skip.
        # ----------------------------------------------------

        if pd.isna(entry_price) or entry_price <= 0:
            continue

        if pd.isna(stop_price) or stop_price <= 0:
            continue

        # ----------------------------------------------------
        # Position sizing uses actual next-day entry.
        # ----------------------------------------------------

        trade_plan = build_trade_plan(
            INITIAL_CAPITAL,
            entry_price,
            signal_atr,
            recent_low,
        )

        # ----------------------------------------------------
        # Targets based on ACTUAL entry.
        # ----------------------------------------------------

        target1 = entry_price * 1.07
        target2 = entry_price * 1.15

        rows.append({
            "Symbol": symbol,

            "SignalDate": signal_date,
            "EntryDate": entry_date,

            "SignalClose": float(signal_close),
            "EntryPrice": float(entry_price),

            "Score": int(signal_row["CANONICAL_SCORE"]),

            "WeeklyTrend": bool(
                signal_row["CANONICAL_WEEKLY_TREND"]
            ),

            "WeeklyMACD": bool(
                signal_row["CANONICAL_WEEKLY_MACD"]
            ),

            "WeeklyConfirm": bool(
                signal_row["CANONICAL_WEEKLY_CONFIRM"]
            ),

            "ATR14": float(signal_atr),

            "RecentLow5": float(recent_low),

            "StopPrice": float(
                trade_plan["stop_price"]
            ),

            "RiskPerShare": float(
                trade_plan["risk_per_share"]
            ),

            "Shares": int(
                trade_plan["shares"]
            ),

            "Lots": int(
                trade_plan["lots"]
            ),

            "RiskAmount": float(
                trade_plan["risk_amount"]
            ),

            "PositionValue": float(
                trade_plan["position_value"]
            ),

            "Target1": float(target1),
            "Target2": float(target2),

            "EntryOpen": float(entry_row["Open"]),
            "EntryHigh": float(entry_row["High"]),
            "EntryLow": float(entry_row["Low"]),
            "EntryClose": float(entry_row["Close"]),
        })

    return pd.DataFrame(rows)


# ============================================================
# GENERATE SIGNALS FOR ALL SYMBOLS
# ============================================================

historical_signals_list = []

for symbol, df in canonical_data.items():

    signals = generate_historical_signals(
        symbol,
        df,
        min_score=SIGNAL_MIN_SCORE,
    )

    if not signals.empty:
        historical_signals_list.append(signals)


if historical_signals_list:
    historical_signals = pd.concat(
        historical_signals_list,
        ignore_index=True,
    )
else:
    historical_signals = pd.DataFrame()


# ============================================================
# SORT
# ============================================================

if not historical_signals.empty:
    historical_signals = historical_signals.sort_values(
        ["SignalDate", "Score"],
        ascending=[True, False],
    ).reset_index(drop=True)


# ============================================================
# SUMMARY
# ============================================================

print("\n====================================")
print("📊 HISTORICAL SIGNAL SUMMARY")
print("====================================")

print(
    f"Total signals : {len(historical_signals):,}"
)

if not historical_signals.empty:

    print(
        f"Symbols       : "
        f"{historical_signals['Symbol'].nunique()}"
    )

    print(
        f"Score >= 3    : "
        f"{(historical_signals['Score'] >= 3).sum():,}"
    )

    print(
        f"Weekly confirm: "
        f"{historical_signals['WeeklyConfirm'].sum():,}"
    )

    print(
        f"Date range    : "
        f"{historical_signals['SignalDate'].min().date()} "
        f"→ "
        f"{historical_signals['SignalDate'].max().date()}"
    )


# ============================================================
# LOOKAHEAD SANITY CHECK
# ============================================================

print("\n====================================")
print("🔬 LOOKAHEAD SANITY CHECK")
print("====================================")

if not historical_signals.empty:

    invalid_entry = (
        historical_signals["EntryDate"]
        <= historical_signals["SignalDate"]
    ).sum()

    invalid_price = (
        historical_signals["EntryPrice"] <= 0
    ).sum()

    invalid_stop = (
        historical_signals["StopPrice"]
        >= historical_signals["SignalClose"]
    ).sum()

    print(
        f"EntryDate <= SignalDate : {invalid_entry}"
    )

    print(
        f"Invalid EntryPrice      : {invalid_price}"
    )

    print(
        f"Invalid Stop             : {invalid_stop}"
    )

    assert invalid_entry == 0
    assert invalid_price == 0
    assert invalid_stop == 0


# ============================================================
# SAMPLE SIGNALS
# ============================================================

print("\n====================================")
print("🔎 SAMPLE HISTORICAL SIGNALS")
print("====================================")

if historical_signals.empty:
    print("No historical signals found.")
else:
    display(
        historical_signals[
            [
                "Symbol",
                "SignalDate",
                "EntryDate",
                "SignalClose",
                "EntryPrice",
                "Score",
                "WeeklyConfirm",
                "ATR14",
                "StopPrice",
                "Shares",
                "RiskAmount",
                "Target1",
                "Target2",
            ]
        ].tail(15)
    )


# ============================================================
# FINAL CHECK
# ============================================================

print("\n====================================")
print("✅ Historical Signal Generation Ready")
print("====================================")

📡 HISTORICAL SIGNAL GENERATION

📊 HISTORICAL SIGNAL SUMMARY
Total signals : 4,222
Symbols       : 19
Score >= 3    : 4,222
Weekly confirm: 1,615
Date range    : 2021-11-02 → 2026-10-05

🔬 LOOKAHEAD SANITY CHECK
EntryDate <= SignalDate : 0
Invalid EntryPrice      : 0
Invalid Stop             : 0

🔎 SAMPLE HISTORICAL SIGNALS


,Symbol,SignalDate,EntryDate,SignalClose,EntryPrice,Score,WeeklyConfirm,ATR14,StopPrice,Shares,RiskAmount,Target1,Target2
4207,AU.BK,2026-09-29,2026-09-30,4.20,4.20,3,False,0.042640,4.136040,23800,1522.248099,4.4940,4.8300
4208,FORTH.BK,2026-09-29,2026-09-30,15.50,15.50,3,False,0.612579,14.400000,1800,1980.000687,16.5850,17.8250
4209,SNNP.BK,2026-09-29,2026-09-30,6.70,6.65,3,False,0.087901,6.518149,15000,1977.767334,7.1155,7.6475
4210,SPA.BK,2026-09-29,2026-09-30,2.96,2.98,3,False,0.067462,2.878807,19700,1993.501259,3.1886,3.4270
4211,PLANB.BK,2026-09-30,2026-10-01,6.95,6.85,4,False,0.291604,5.750000,1800,1979.999828,7.3295,7.8775
4212,AU.BK,2026-09-30,2026-10-01,4.16,4.16,3,False,0.042451,4.096323,24000,1528.251424,4.4512,4.7840
4213,FORTH.BK,2026-09-30,2026-10-01,14.60,14.40,3,False,0.647395,13.428908,2000,1942.183787,15.4080,16.5600
4214,SNNP.BK,2026-09-30,2026-10-01,6.65,6.60,3,False,0.095908,6.456138,13900,1999.678560,7.0620,7.5900
4215,TKN.BK,2026-09-30,2026-10-01,4.70,4.70,3,True,0.103949,4.440000,7600,1975.998116,5.0290,5.4050
4216,SPA.BK,2026-10-02,2026-10-05,2.98,2.98,4,False,0.067403,2.878895,19700,1991.759559,3.1886,3.4270



✅ Historical Signal Generation Ready


In [53]:

# ============================================================
# 11.2) FIX HISTORICAL SIGNALS
# PRESERVE ORIGINAL SIGNAL SET
# FIX STOP FROM SIGNAL-DATE DATA
# ============================================================

print("🔧 FIXING HISTORICAL SIGNALS")
print("=" * 90)

old_signals = historical_signals.copy()

print(f"Original signals : {len(old_signals):,}")

fixed_rows = []
excluded_invalid_stop = 0

for _, old in old_signals.iterrows():

    symbol = old["Symbol"]
    signal_date = pd.Timestamp(old["SignalDate"])
    entry_date = pd.Timestamp(old["EntryDate"])

    if symbol not in feature_data:
        continue

    df = feature_data[symbol].sort_index()

    if signal_date not in df.index:
        continue

    if entry_date not in df.index:
        continue

    # RECENT_LOW_5 is created on canonical_data in Cell 9.
    # feature_data may not carry that derived column, so rebuild it
    # here from signal-date history if it is missing.
    if "RECENT_LOW_5" not in df.columns:
        df = df.copy()
        df["RECENT_LOW_5"] = (
            df["Low"]
            .rolling(
                window=RECENT_LOW_LOOKBACK,
                min_periods=RECENT_LOW_LOOKBACK,
            )
            .min()
        )

    signal_row = df.loc[signal_date]
    entry_row = df.loc[entry_date]

    signal_close = float(signal_row["Close"])
    signal_atr = float(signal_row["ATR14"])
    recent_low_value = signal_row["RECENT_LOW_5"]

    if not np.isfinite(recent_low_value) or recent_low_value <= 0:
        continue

    recent_low = float(recent_low_value)
    entry_price = float(entry_row["Open"])

    # --------------------------------------------------------
    # FIXED STOP FROM SIGNAL DATE
    # --------------------------------------------------------

    stop_price = calculate_stop_price(
        entry_price=signal_close,
        atr=signal_atr,
        recent_low=recent_low,
    )

    if not np.isfinite(stop_price) or stop_price <= 0:
        excluded_invalid_stop += 1
        continue

    # A long trade cannot have its protective stop at or above
    # the actual next-day entry price. This can happen after a
    # gap-down between signal date and entry date. Exclude that
    # trade from the executable historical set rather than
    # manufacturing a stop or silently changing the strategy.
    if entry_price <= stop_price:
        excluded_invalid_stop += 1
        continue

    # --------------------------------------------------------
    # POSITION SIZE
    # Uses actual entry price + fixed stop
    # --------------------------------------------------------

    risk_per_share = entry_price - stop_price

    if risk_per_share <= 0:
        shares = 0
        lots = 0
        risk_amount = 0.0
        position_value = 0.0
        status = "INVALID_STOP"

    else:
        max_risk = INITIAL_CAPITAL * MAX_RISK_PCT

        risk_shares = int(max_risk / risk_per_share)
        capital_shares = int(INITIAL_CAPITAL / entry_price)

        shares = min(risk_shares, capital_shares)

        shares = (shares // LOT_SIZE) * LOT_SIZE
        lots = shares // LOT_SIZE

        risk_amount = risk_per_share * shares
        position_value = entry_price * shares

        status = "OK" if shares > 0 else "CAPITAL_LIMITED"

    # --------------------------------------------------------
    # TARGETS
    # --------------------------------------------------------

    target1 = entry_price * 1.07
    target2 = entry_price * 1.15

    fixed_rows.append({
        "Symbol": symbol,
        "SignalDate": signal_date,
        "EntryDate": entry_date,
        "SignalClose": signal_close,
        "EntryPrice": entry_price,
        "Score": int(old["Score"]),
        "WeeklyConfirm": bool(old["WeeklyConfirm"]),
        "ATR14": signal_atr,
        "RecentLow5": recent_low,
        "StopPrice": stop_price,
        "Shares": shares,
        "Lots": lots,
        "RiskAmount": risk_amount,
        "PositionValue": position_value,
        "Target1": target1,
        "Target2": target2,
        "Status": status,
        "EntryOpen": float(entry_row["Open"]),
        "EntryHigh": float(entry_row["High"]),
        "EntryLow": float(entry_row["Low"]),
        "EntryClose": float(entry_row["Close"]),
    })


# ------------------------------------------------------------
# SAFETY CHECK
# ------------------------------------------------------------

if not fixed_rows:
    raise RuntimeError(
        "No signals were rebuilt. Stop here and inspect Cell 11."
    )


historical_signals = pd.DataFrame(fixed_rows)

historical_signals = (
    historical_signals
    .sort_values(["SignalDate", "Symbol"])
    .reset_index(drop=True)
)


# ============================================================
# SUMMARY
# ============================================================

print("\n====================================")
print("📊 REBUILD SUMMARY")
print("====================================")

print(f"Original signals : {len(old_signals):,}")
print(f"Fixed signals    : {len(historical_signals):,}")
print(
    f"Difference       : "
    f"{len(historical_signals) - len(old_signals):+,}"
)


# ============================================================
# VALIDATION
# ============================================================

entry_before_signal = (
    historical_signals["EntryDate"]
    < historical_signals["SignalDate"]
).sum()

invalid_entry = (
    ~np.isfinite(historical_signals["EntryPrice"])
    | (historical_signals["EntryPrice"] <= 0)
).sum()

invalid_stop = (
    ~np.isfinite(historical_signals["StopPrice"])
    | (historical_signals["StopPrice"] <= 0)
).sum()

stop_ge_entry = (
    historical_signals["StopPrice"]
    >= historical_signals["EntryPrice"]
).sum()

risk_over_limit = (
    historical_signals["RiskAmount"]
    > INITIAL_CAPITAL * MAX_RISK_PCT + 1e-9
).sum()

capital_over_limit = (
    historical_signals["PositionValue"]
    > INITIAL_CAPITAL + 1e-9
).sum()


print("\n====================================")
print("🔍 VALIDATION")
print("====================================")

print(f"Entry before Signal : {entry_before_signal:,}")
print(f"Invalid EntryPrice  : {invalid_entry:,}")
print(f"Invalid Stop        : {invalid_stop:,}")
print(f"Stop >= Entry       : {stop_ge_entry:,}")
print(f"Risk > 2%           : {risk_over_limit:,}")
print(f"Capital > 100k      : {capital_over_limit:,}")


# ============================================================
# STOP CONSISTENCY
# ============================================================

check = historical_signals.head(10).copy()

check["ExpectedStop"] = check.apply(
    lambda r: calculate_stop_price(
        r["SignalClose"],
        r["ATR14"],
        r["RecentLow5"],
    ),
    axis=1,
)

check["StopMatch"] = np.isclose(
    check["StopPrice"],
    check["ExpectedStop"],
    rtol=1e-10,
    atol=1e-10,
)

print("\n====================================")
print("🔒 STOP CONSISTENCY")
print("====================================")

display(
    check[
        [
            "Symbol",
            "SignalDate",
            "SignalClose",
            "EntryPrice",
            "ATR14",
            "RecentLow5",
            "StopPrice",
            "ExpectedStop",
            "StopMatch",
        ]
    ]
)


# ============================================================
# FINAL STATUS
# ============================================================

print("\n====================================")

assert len(historical_signals) + excluded_invalid_stop == len(old_signals), (
    "Signal accounting mismatch during rebuild."
)
assert entry_before_signal == 0, "EntryDate is before SignalDate."
assert invalid_entry == 0, "Invalid EntryPrice found."
assert invalid_stop == 0, "Invalid StopPrice found."
assert stop_ge_entry == 0, "StopPrice >= EntryPrice found."
assert risk_over_limit == 0, "Risk exceeds configured limit."
assert capital_over_limit == 0, "Position value exceeds initial capital."
assert check["StopMatch"].all(), "Stop-price consistency check failed."

print("✅ HISTORICAL SIGNALS FIXED")
print("CELL 11.2 PASSED")
print("====================================")

🔧 FIXING HISTORICAL SIGNALS
Original signals : 4,222

📊 REBUILD SUMMARY
Original signals : 4,222
Fixed signals    : 4,217
Difference       : -5

🔍 VALIDATION
Entry before Signal : 0
Invalid EntryPrice  : 0
Invalid Stop        : 0
Stop >= Entry       : 0
Risk > 2%           : 0
Capital > 100k      : 0

🔒 STOP CONSISTENCY


,Symbol,SignalDate,SignalClose,EntryPrice,ATR14,RecentLow5,StopPrice,ExpectedStop,StopMatch
0,KAMART.BK,2021-11-02,3.377142,3.325714,0.062393,3.222857,3.222857,3.222857,True
1,PLANB.BK,2021-11-03,6.157406,6.157406,0.206672,6.108538,5.847399,5.847399,True
2,AU.BK,2021-11-04,10.300000,10.300000,0.200752,10.200000,9.998872,9.998872,True
3,CBG.BK,2021-11-04,125.500000,126.000000,2.693017,121.500000,121.460475,121.460475,True
4,DELTA.BK,2021-11-04,44.599998,45.000000,1.999531,40.799999,40.799999,40.799999,True
5,PLANB.BK,2021-11-04,6.499484,6.499484,0.223325,6.108538,6.108538,6.108538,True
6,CBG.BK,2021-11-05,125.000000,125.000000,2.714944,121.500000,120.927584,120.927584,True
7,DELTA.BK,2021-11-05,45.200001,45.599998,2.085279,40.799999,40.799999,40.799999,True
8,JMT.BK,2021-11-05,49.750000,49.000000,1.361534,45.915836,45.915836,45.915836,True
9,XO.BK,2021-11-05,17.100000,17.400000,0.484075,16.900000,16.373888,16.373888,True



✅ HISTORICAL SIGNALS FIXED
CELL 11.2 PASSED


In [54]:

# ============================================================
# 12.1) SIGNAL CLUSTER ANALYSIS
# NO TRADE-ENGINE DEPENDENCY
# ============================================================

print("🔍 SIGNAL CLUSTER ANALYSIS")
print("=" * 90)

assert not historical_signals.empty, "historical_signals is empty."

analysis = historical_signals.copy()
analysis["SignalDate"] = pd.to_datetime(analysis["SignalDate"])
analysis["EntryDate"] = pd.to_datetime(analysis["EntryDate"])

analysis = analysis.sort_values(
    ["Symbol", "SignalDate"]
).reset_index(drop=True)

# ------------------------------------------------------------
# Days since previous signal for the same symbol
# ------------------------------------------------------------

analysis["PrevSignalDate"] = (
    analysis.groupby("Symbol")["SignalDate"].shift(1)
)

analysis["DaysSincePrevSignal"] = (
    analysis["SignalDate"] - analysis["PrevSignalDate"]
).dt.days

analysis["ClusteredSignal"] = (
    analysis["DaysSincePrevSignal"].notna()
    & (analysis["DaysSincePrevSignal"] <= 3)
)

clustered = analysis[analysis["ClusteredSignal"]].copy()

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

total_signals = len(analysis)
clustered_count = len(clustered)
cluster_rate = clustered_count / total_signals * 100

print("\n====================================")
print("📊 SIGNAL CLUSTER SUMMARY")
print("====================================")
print(f"Total signals     : {total_signals:,}")
print(f"Clustered signals : {clustered_count:,}")
print(f"Cluster rate      : {cluster_rate:.2f}%")

# ------------------------------------------------------------
# Symbol-level summary
# ------------------------------------------------------------

symbol_cluster = (
    analysis.groupby("Symbol")
    .agg(
        Signals=("SignalDate", "count"),
        Clustered=("ClusteredSignal", "sum"),
    )
)

symbol_cluster["ClusterRate"] = (
    symbol_cluster["Clustered"]
    / symbol_cluster["Signals"]
    * 100
)

symbol_cluster = symbol_cluster.sort_values(
    ["Signals", "ClusterRate"],
    ascending=[False, False],
)

print("\n====================================")
print("📈 SIGNALS BY SYMBOL")
print("====================================")
display(symbol_cluster)

# ------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------

same_symbol_order_ok = (
    analysis.groupby("Symbol")["SignalDate"]
    .apply(lambda s: s.is_monotonic_increasing)
    .all()
)

invalid_gap = (
    analysis["DaysSincePrevSignal"].notna()
    & (analysis["DaysSincePrevSignal"] < 0)
).sum()

assert same_symbol_order_ok, "Signal dates are not chronological by symbol."
assert invalid_gap == 0, "Negative signal-date gap found."
assert clustered_count <= total_signals

print("\n====================================")
print("🔎 RECENT CLUSTERED SIGNALS")
print("====================================")

if clustered.empty:
    print("No clustered signals found.")
else:
    display(
        clustered[
            [
                "Symbol",
                "SignalDate",
                "EntryDate",
                "Score",
                "WeeklyConfirm",
                "DaysSincePrevSignal",
            ]
        ].tail(30)
    )

print("\n====================================")
print("✅ CELL 12.1 PASSED")
print("====================================")

🔍 SIGNAL CLUSTER ANALYSIS

📊 SIGNAL CLUSTER SUMMARY
Total signals     : 4,217
Clustered signals : 2,276
Cluster rate      : 53.97%

📈 SIGNALS BY SYMBOL


,Signals,Clustered,ClusterRate
Symbol,,,
SAPPE.BK,277,167,60.288809
KAMART.BK,271,155,57.195572
AU.BK,269,150,55.762082
ICHI.BK,261,147,56.321839
SPA.BK,260,137,52.692308
PLANB.BK,249,131,52.610442
TKN.BK,242,131,54.132231
DITTO.BK,227,114,50.220264
DELTA.BK,223,113,50.672646



🔎 RECENT CLUSTERED SIGNALS


,Symbol,SignalDate,EntryDate,Score,WeeklyConfirm,DaysSincePrevSignal
4164,XO.BK,2025-06-26,2025-06-27,4,False,1.0
4165,XO.BK,2025-06-27,2025-06-30,3,False,1.0
4168,XO.BK,2025-07-18,2025-07-21,3,True,1.0
4174,XO.BK,2025-11-28,2025-12-01,3,False,1.0
4176,XO.BK,2026-01-06,2026-01-07,3,False,1.0
4178,XO.BK,2026-01-21,2026-01-22,3,False,1.0
4180,XO.BK,2026-02-10,2026-02-11,3,False,1.0
4181,XO.BK,2026-02-11,2026-02-12,4,False,1.0
4184,XO.BK,2026-03-17,2026-03-18,3,False,1.0
4186,XO.BK,2026-03-31,2026-04-01,3,False,1.0



✅ CELL 12.1 PASSED


In [ ]:

# ============================================================
# CELL 12.2 — NON-OVERLAPPING BACKTEST
# ============================================================

print("=" * 90)
print("🔬 NON-OVERLAPPING HISTORICAL BACKTEST")
print("=" * 90)

# ------------------------------------------------------------
# 1) Conservative trade simulator
# ------------------------------------------------------------

def simulate_trade_fixed_stop(signal, price_data):
    entry_date = pd.Timestamp(signal["EntryDate"])
    entry_price = float(signal["EntryPrice"])
    stop_price = float(signal["StopPrice"])
    shares = int(signal["Shares"])

    target1 = entry_price * 1.07
    target2 = entry_price * 1.15

    future = price_data.loc[price_data.index >= entry_date].copy()

    if future.empty:
        return None

    target1_hit = False
    exit_date = pd.NaT
    exit_price = np.nan
    exit_reason = None

    for current_date, bar in future.iterrows():

        day_open = float(bar["Open"])
        day_high = float(bar["High"])
        day_low = float(bar["Low"])
        day_close = float(bar["Close"])

        # ----------------------------------------------------
        # Gap handling
        # ----------------------------------------------------
        # If market opens below stop, actual exit is OPEN.
        # This is conservative and avoids assuming execution
        # at a better price than the market provided.
        if day_open <= stop_price:
            exit_date = current_date
            exit_price = day_open
            exit_reason = "STOP_GAP"
            break

        # ----------------------------------------------------
        # Normal intraday stop
        # ----------------------------------------------------
        if day_low <= stop_price:
            exit_date = current_date
            exit_price = stop_price
            exit_reason = "STOP"
            break

        # ----------------------------------------------------
        # Target 2
        # ----------------------------------------------------
        if day_open >= target2:
            target1_hit = True
            exit_date = current_date
            exit_price = day_open
            exit_reason = "TARGET2_GAP"
            break

        if day_high >= target2:
            target1_hit = True
            exit_date = current_date
            exit_price = target2
            exit_reason = "TARGET2"
            break

        # ----------------------------------------------------
        # Target 1 — tracking only
        # ----------------------------------------------------
        if day_high >= target1:
            target1_hit = True

    # --------------------------------------------------------
    # End of data
    # --------------------------------------------------------

    if pd.isna(exit_date):
        exit_date = future.index[-1]
        exit_price = float(future.iloc[-1]["Close"])
        exit_reason = "EOD"

    pnl = (exit_price - entry_price) * shares
    return_pct = ((exit_price - entry_price) / entry_price) * 100

    min_low = float(future["Low"].min())
    max_high = float(future["High"].max())

    mae_pct = ((min_low - entry_price) / entry_price) * 100
    mfe_pct = ((max_high - entry_price) / entry_price) * 100

    return {
        "Symbol": signal["Symbol"],
        "SignalDate": signal["SignalDate"],
        "EntryDate": entry_date,
        "EntryPrice": entry_price,
        "StopPrice": stop_price,
        "Shares": shares,
        "Target1": target1,
        "Target2": target2,
        "Target1Hit": target1_hit,
        "ExitDate": exit_date,
        "ExitPrice": exit_price,
        "ExitReason": exit_reason,
        "PnL": pnl,
        "ReturnPct": return_pct,
        "MAE_Pct": mae_pct,
        "MFE_Pct": mfe_pct,
        "Score": signal["Score"],
        "WeeklyConfirm": signal["WeeklyConfirm"],
    }


# ------------------------------------------------------------
# 2) Backtest one symbol without overlapping trades
# ------------------------------------------------------------

def backtest_one_symbol_nonoverlap(symbol):
    signals = historical_signals_valid[
        historical_signals_valid["Symbol"] == symbol
    ].copy()

    signals = signals.sort_values(
        ["EntryDate", "SignalDate"]
    ).reset_index(drop=True)

    if symbol not in feature_data:
        return pd.DataFrame()

    price_data = feature_data[symbol].copy()

    trades = []
    previous_exit_date = pd.NaT
    skipped_overlap = 0

    for _, signal in signals.iterrows():

        entry_date = pd.Timestamp(signal["EntryDate"])

        # ----------------------------------------------------
        # Prevent overlapping trades
        # ----------------------------------------------------
        if not pd.isna(previous_exit_date):
            if entry_date <= previous_exit_date:
                skipped_overlap += 1
                continue

        trade = simulate_trade_fixed_stop(
            signal,
            price_data
        )

        if trade is None:
            continue

        trades.append(trade)
        previous_exit_date = trade["ExitDate"]

    result = pd.DataFrame(trades)

    return result


# ------------------------------------------------------------
# 3) Test AU.BK first
# ------------------------------------------------------------

TEST_SYMBOL = "AU.BK"

au_trades = backtest_one_symbol_nonoverlap(TEST_SYMBOL)

print("\n====================================")
print(f"📊 {TEST_SYMBOL} BACKTEST")
print("====================================")

print(f"Signals available : {len(historical_signals_valid[historical_signals_valid['Symbol'] == TEST_SYMBOL]):,}")
print(f"Trades executed   : {len(au_trades):,}")

if not au_trades.empty:

    print(f"Total P&L         : {au_trades['PnL'].sum():,.2f} THB")
    print(f"Average Return    : {au_trades['ReturnPct'].mean():.2f}%")
    print(f"Median Return     : {au_trades['ReturnPct'].median():.2f}%")
    print(f"Win Rate          : {(au_trades['ReturnPct'] > 0).mean() * 100:.2f}%")

    print("\n====================================")
    print("🔎 TRADE SAMPLE")
    print("====================================")

    display(
        au_trades[
            [
                "Symbol",
                "SignalDate",
                "EntryDate",
                "EntryPrice",
                "StopPrice",
                "ExitDate",
                "ExitPrice",
                "ExitReason",
                "Shares",
                "PnL",
                "ReturnPct",
                "Target1Hit",
            ]
        ].head(15)
    )

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    invalid_exit = (
        au_trades["ExitDate"] < au_trades["EntryDate"]
    ).sum()

    invalid_return = (
        ~pd.to_numeric(
            au_trades["ReturnPct"],
            errors="coerce"
        ).notna()
    ).sum()

    print("\n====================================")
    print("🛡️ BACKTEST VALIDATION")
    print("====================================")
    print(f"Exit before Entry : {invalid_exit}")
    print(f"Invalid Return    : {invalid_return}")

    assert invalid_exit == 0
    assert invalid_return == 0

    print("\n====================================")
    print("✅ CELL 12.2 PASSED")
    print("====================================")
    print("Non-overlapping backtest is valid for the test symbol.")

else:
    print("\n⚠️ No executable trades found for", TEST_SYMBOL)

## CHECKPOINT — 12.1 PASSED

Cell 12.1 has been validated. Run **Cell 12.2 only** next.

If Cell 12.2 prints `CELL 12.2 PASSED`, the next stage can be added.